# NeuroGolf Sparse Builder

This notebook is the shareable pure-code version of my local NeuroGolf pipeline.

Expected behavior: this notebook should produce a valid sparse submission from code alone. The LB may differ from my merged local submission because that one used an accepted reference artifact as a base.


In [ ]:
!pip install -q onnxruntime onnx-tool


In [ ]:
import csv
import importlib.util
import json
import math
import tempfile
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import onnx
import onnx.helper as oh
import onnx.numpy_helper as onh
import onnxruntime as ort


def resolve_task_dir() -> Path:
    candidates = [
        Path('/kaggle/input/competitions/neurogolf-2026'),
        Path('/kaggle/input/neurogolf-2026'),
        Path('/kaggle/input'),
        Path('.'),
    ]
    for candidate in candidates:
        if candidate.exists() and list(candidate.glob('task*.json')):
            return candidate
    for root in candidates:
        if not root.exists():
            continue
        for path in root.rglob('task001.json'):
            return path.parent
    raise FileNotFoundError('Could not find task*.json files')


TASK_DIR = resolve_task_dir()
WORKDIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
print('TASK_DIR =', TASK_DIR)
print('WORKDIR =', WORKDIR)


In [ ]:
%%writefile starter_solver.py
import os, json, zipfile, copy, itertools
import numpy as np
from pathlib import Path
from collections import defaultdict
from scipy.ndimage import label as scipy_label

import onnx
import onnx.helper as oh
import onnx.numpy_helper as onh
from onnx import TensorProto
TASK_DIR = Path("/kaggle/input/competitions/neurogolf-2026")
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
C, H, W = 10, 30, 30

def load_task(path):
    with open(path) as f:
        return json.load(f)


def grid_to_tensor(grid):
    g = np.array(grid, dtype=np.int32)
    h, w = g.shape
    t = np.zeros((1, C, H, W), dtype=np.float32)
    for r in range(h):
        for c in range(w):
            v = g[r, c]
            if 0 <= v <= 9:
                t[0, v, r, c] = 1.0
    return t


def tensor_to_grid(t, out_h, out_w):
    arr = t[0]
    grid = []
    for r in range(out_h):
        row = []
        for c in range(out_w):
            col_vals = arr[:, r, c]
            if col_vals.max() < 0.5:
                row.append(0)
            else:
                row.append(int(np.argmax(col_vals)))
        grid.append(row)
    return grid


def check_model_correct(model, pairs):
    try:
        import onnxruntime as ort
    except ImportError:
        os.system("pip install onnxruntime -q")
        import onnxruntime as ort
    try:
        buf = model.SerializeToString()
        sess = ort.InferenceSession(buf, providers=["CPUExecutionProvider"])
        for p in pairs:
            inp = grid_to_tensor(p["input"])
            out_g = np.array(p["output"])
            oh_, ow_ = out_g.shape
            pred = sess.run(None, {"input": inp})[0]
            pred_grid = tensor_to_grid(pred, oh_, ow_)
            if pred_grid != p["output"]:
                return False
        return True
    except Exception:
        return False


def make_onnx_from_gather(gather_indices, graph_name="permute"):
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    shape_chw = onh.from_array(np.array([1, C, H * W], dtype=np.int64), name="shape_chw")
    shape_chw_node = oh.make_node("Constant", [], ["shape_chw"], value=shape_chw)
    inp_reshape = oh.make_node("Reshape", ["input", "shape_chw"], ["inp_chw"])
    gi_tensor = onh.from_array(gather_indices.astype(np.int64), name="gather_idx")
    gi_node = oh.make_node("Constant", [], ["gather_idx"], value=gi_tensor)
    gather_node = oh.make_node("Gather", ["inp_chw", "gather_idx"], ["gathered"], axis=2)
    out_shape = onh.from_array(np.array([1, C, H, W], dtype=np.int64), name="out_shape")
    out_shape_node = oh.make_node("Constant", [], ["out_shape"], value=out_shape)
    reshape_back = oh.make_node("Reshape", ["gathered", "out_shape"], ["output"])
    nodes = [shape_chw_node, inp_reshape, gi_node, gather_node, out_shape_node, reshape_back]
    graph = oh.make_graph(nodes, graph_name, [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model


def make_identity_onnx():
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    node = oh.make_node("Identity", ["input"], ["output"])
    graph = oh.make_graph([node], "identity", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model


def make_const_onnx(const_output):
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    const_val = const_output.astype(np.float32)
    const_tensor = onh.from_array(const_val, name="const_out")
    const_node = oh.make_node("Constant", [], ["const_out"], value=const_tensor)
    zero_tensor = onh.from_array(np.zeros((1, C, H, W), dtype=np.float32), name="zero_val")
    zero_node = oh.make_node("Constant", [], ["zero_val"], value=zero_tensor)
    mul_node = oh.make_node("Mul", ["input", "zero_val"], ["zeroed"])
    add_node = oh.make_node("Add", ["zeroed", "const_out"], ["output"])
    graph = oh.make_graph([const_node, zero_node, mul_node, add_node], "const_net", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model


def make_color_remap_onnx(color_map):
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    weight = np.zeros((C, C, 1, 1), dtype=np.float32)
    for src, dst in color_map.items():
        if 0 <= src < C and 0 <= dst < C:
            weight[dst, src, 0, 0] = 1.0
    for ch in range(C):
        if ch not in color_map and not any(v == ch for v in color_map.values()):
            weight[ch, ch, 0, 0] = 1.0
    w_tensor = onh.from_array(weight, name="conv_weight")
    w_node = oh.make_node("Constant", [], ["conv_weight"], value=w_tensor)
    conv_node = oh.make_node("Conv", ["input", "conv_weight"], ["output"],
                             kernel_shape=[1, 1], pads=[0, 0, 0, 0])
    nodes = [w_node, conv_node]
    graph = oh.make_graph(nodes, "color_remap", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model


def make_conv_onnx(weight_np, kernel_size, graph_name="conv_net"):
    pad = kernel_size // 2
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    w_tensor = onh.from_array(weight_np.astype(np.float32), name="conv_weight")
    w_node = oh.make_node("Constant", [], ["conv_weight"], value=w_tensor)
    conv_node = oh.make_node("Conv", ["input", "conv_weight"], ["output"],
                             kernel_shape=[kernel_size, kernel_size],
                             pads=[pad, pad, pad, pad])
    nodes = [w_node, conv_node]
    graph = oh.make_graph(nodes, graph_name, [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model


def make_masked_gather_onnx(gather_indices, mask, graph_name="masked_gather"):
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    shape_chw = onh.from_array(np.array([1, C, H * W], dtype=np.int64), name="shape_chw")
    shape_chw_node = oh.make_node("Constant", [], ["shape_chw"], value=shape_chw)
    inp_reshape = oh.make_node("Reshape", ["input", "shape_chw"], ["inp_chw"])
    gi_tensor = onh.from_array(gather_indices.astype(np.int64), name="gather_idx")
    gi_node = oh.make_node("Constant", [], ["gather_idx"], value=gi_tensor)
    gather_node = oh.make_node("Gather", ["inp_chw", "gather_idx"], ["gathered"], axis=2)
    out_shape = onh.from_array(np.array([1, C, H, W], dtype=np.int64), name="out_shape")
    out_shape_node = oh.make_node("Constant", [], ["out_shape"], value=out_shape)
    reshape_back = oh.make_node("Reshape", ["gathered", "out_shape"], ["gathered_4d"])
    mask_tensor = onh.from_array(mask.astype(np.float32), name="mask")
    mask_node = oh.make_node("Constant", [], ["mask"], value=mask_tensor)
    mul_node = oh.make_node("Mul", ["gathered_4d", "mask"], ["output"])
    nodes = [shape_chw_node, inp_reshape, gi_node, gather_node,
             out_shape_node, reshape_back, mask_node, mul_node]
    graph = oh.make_graph(nodes, graph_name, [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model


def gather_indices_for_transform(transform_fn):
    indices = np.zeros(H * W, dtype=np.int64)
    for r in range(H):
        for c in range(W):
            nr, nc = transform_fn(r, c)
            nr = max(0, min(H - 1, nr))
            nc = max(0, min(W - 1, nc))
            indices[r * W + c] = nr * W + nc
    return indices


def detect_rotation(pairs):
    for angle in [90, 180, 270]:
        k = angle // 90
        valid = True
        for p in pairs:
            ig = np.array(p["input"])
            og = np.array(p["output"])
            if np.rot90(ig, k).shape != og.shape or not np.array_equal(np.rot90(ig, k), og):
                valid = False
                break
        if valid:
            return angle
    return None


def make_rotation_onnx(angle):
    k = angle // 90
    if k == 1:
        fn = lambda r, c: (c, H - 1 - r)
    elif k == 2:
        fn = lambda r, c: (H - 1 - r, W - 1 - c)
    elif k == 3:
        fn = lambda r, c: (W - 1 - c, r)
    else:
        return make_identity_onnx()
    return make_onnx_from_gather(gather_indices_for_transform(fn), "rotation_net")


def detect_flip(pairs):
    for axis, name in [(0, "vertical"), (1, "horizontal"), (-1, "both")]:
        valid = True
        for p in pairs:
            ig, og = np.array(p["input"]), np.array(p["output"])
            flipped = np.flip(ig, axis=(0, 1) if axis == -1 else axis)
            if not np.array_equal(flipped, og):
                valid = False
                break
        if valid:
            return name
    return None


def make_flip_onnx(direction):
    if direction == "vertical":
        fn = lambda r, c: (H - 1 - r, c)
    elif direction == "horizontal":
        fn = lambda r, c: (r, W - 1 - c)
    else:
        fn = lambda r, c: (H - 1 - r, W - 1 - c)
    return make_onnx_from_gather(gather_indices_for_transform(fn), "flip_net")


def detect_transpose(pairs):
    for p in pairs:
        ig, og = np.array(p["input"]), np.array(p["output"])
        if ig.T.shape != og.shape or not np.array_equal(ig.T, og):
            return False
    return True


def make_transpose_onnx():
    fn = lambda r, c: (c, r)
    return make_onnx_from_gather(gather_indices_for_transform(fn), "transpose_net")


def detect_color_map(pairs):
    if not all(np.array(p["input"]).shape == np.array(p["output"]).shape for p in pairs):
        return None
    color_maps = []
    for p in pairs:
        ig = np.array(p["input"]).flatten()
        og = np.array(p["output"]).flatten()
        cmap = {}
        valid = True
        for a, b in zip(ig, og):
            if a in cmap and cmap[a] != b:
                valid = False
                break
            cmap[a] = b
        if not valid:
            return None
        color_maps.append(cmap)
    if not color_maps:
        return None
    common = color_maps[0]
    for cm in color_maps[1:]:
        if cm != common:
            return None
    return common


def detect_crop(pairs):
    results = []
    for p in pairs:
        ig, og = np.array(p["input"]), np.array(p["output"])
        ih, iw = ig.shape
        oh_, ow_ = og.shape
        if oh_ > ih or ow_ > iw:
            return None
        found = False
        for r0 in range(ih - oh_ + 1):
            for c0 in range(iw - ow_ + 1):
                if np.array_equal(ig[r0:r0 + oh_, c0:c0 + ow_], og):
                    results.append((r0, c0))
                    found = True
                    break
            if found:
                break
        if not found:
            return None
    if len(set(results)) == 1:
        r0, c0 = results[0]
        oh_ = np.array(pairs[0]["output"]).shape[0]
        ow_ = np.array(pairs[0]["output"]).shape[1]
        return r0, c0, oh_, ow_
    return None


def make_crop_onnx(r0, c0, oh_, ow_):
    gather_indices = np.zeros(H * W, dtype=np.int64)
    for r in range(H):
        for c_ in range(W):
            if r < oh_ and c_ < ow_:
                src_r = min(r0 + r, H - 1)
                src_c = min(c0 + c_, W - 1)
                gather_indices[r * W + c_] = src_r * W + src_c
    if oh_ < H or ow_ < W:
        mask = np.zeros((1, C, H, W), dtype=np.float32)
        mask[0, :, :oh_, :ow_] = 1.0
        return make_masked_gather_onnx(gather_indices, mask, "crop_net")
    return make_onnx_from_gather(gather_indices, "crop_net")


def detect_tile(pairs):
    for p in pairs:
        ig, og = np.array(p["input"]), np.array(p["output"])
        ih, iw = ig.shape
        oh_, ow_ = og.shape
        if oh_ % ih != 0 or ow_ % iw != 0:
            return None
        tr, tc = oh_ // ih, ow_ // iw
        if not np.array_equal(np.tile(ig, (tr, tc)), og):
            return None
    ih, iw = np.array(pairs[0]["input"]).shape
    oh_, ow_ = np.array(pairs[0]["output"]).shape
    return oh_ // ih, ow_ // iw


def make_tile_onnx(tile_r, tile_c):
    ih = H // tile_r if H % tile_r == 0 else H
    iw = W // tile_c if W % tile_c == 0 else W
    gather_indices = np.zeros(H * W, dtype=np.int64)
    mask = np.zeros((1, C, H, W), dtype=np.float32)
    oh_ = ih * tile_r
    ow_ = iw * tile_c
    for r in range(H):
        for c in range(W):
            if r < oh_ and c < ow_:
                src_r = r % ih
                src_c = c % iw
                gather_indices[r * W + c] = src_r * W + src_c
                mask[0, :, r, c] = 1.0
    if oh_ < H or ow_ < W:
        return make_masked_gather_onnx(gather_indices, mask, "tile_net")
    return make_onnx_from_gather(gather_indices, "tile_net")


def detect_scale(pairs):
    for factor in [2, 3, 4, 5]:
        valid = True
        for p in pairs:
            ig, og = np.array(p["input"]), np.array(p["output"])
            ih, iw = ig.shape
            oh_, ow_ = og.shape
            if oh_ != ih * factor or ow_ != iw * factor:
                valid = False
                break
            scaled = np.repeat(np.repeat(ig, factor, axis=0), factor, axis=1)
            if not np.array_equal(scaled, og):
                valid = False
                break
        if valid:
            return factor
    return None


def make_scale_onnx(factor):
    gather_indices = np.zeros(H * W, dtype=np.int64)
    mask = np.zeros((1, C, H, W), dtype=np.float32)
    for p_ih in range(1, 31):
        for p_iw in range(1, 31):
            oh_ = p_ih * factor
            ow_ = p_iw * factor
            if oh_ <= H and ow_ <= W:
                for r in range(oh_):
                    for c in range(ow_):
                        src_r = r // factor
                        src_c = c // factor
                        gather_indices[r * W + c] = src_r * W + src_c
                        mask[0, :, r, c] = 1.0
                break
        break
    for r in range(H):
        for c in range(W):
            if mask[0, 0, r, c] < 0.5:
                gather_indices[r * W + c] = 0

    gather_indices_final = np.zeros(H * W, dtype=np.int64)
    mask_final = np.zeros((1, C, H, W), dtype=np.float32)
    for r in range(H):
        for c in range(W):
            src_r = (r // factor)
            src_c = (c // factor)
            if src_r < H and src_c < W:
                gather_indices_final[r * W + c] = src_r * W + src_c
    return make_onnx_from_gather(gather_indices_final, "scale_net")


def detect_pixel_permutation(pairs):
    if not all(np.array(p["input"]).shape == np.array(p["output"]).shape for p in pairs):
        return None
    first = pairs[0]
    ig = np.array(first["input"])
    og = np.array(first["output"])
    ih, iw = ig.shape
    if ih != H or iw != W:
        return None
    src_flat = ig.flatten()
    dst_flat = og.flatten()
    assigned = {}
    for dst_idx in range(H * W):
        dst_val = dst_flat[dst_idx]
        matches = np.where(src_flat == dst_val)[0]
        if len(matches) == 1:
            assigned[dst_idx] = matches[0]
        else:
            return None
    gi = np.array([assigned.get(i, i) for i in range(H * W)], dtype=np.int64)
    for p in pairs[1:]:
        ig2 = np.array(p["input"]).flatten()
        og2 = np.array(p["output"]).flatten()
        if not np.array_equal(ig2[gi], og2):
            return None
    return gi


def detect_gravity(pairs, direction="down"):
    for p in pairs:
        ig, og = np.array(p["input"]), np.array(p["output"])
        if ig.shape != og.shape:
            return False
        ih, iw = ig.shape
        expected = np.zeros_like(ig)
        if direction == "down":
            for c in range(iw):
                col = ig[:ih, c]
                nonzero = col[col != 0]
                expected[ih - len(nonzero):ih, c] = nonzero
        elif direction == "up":
            for c in range(iw):
                col = ig[:ih, c]
                nonzero = col[col != 0]
                expected[:len(nonzero), c] = nonzero
        elif direction == "right":
            for r in range(ih):
                row = ig[r, :iw]
                nonzero = row[row != 0]
                expected[r, iw - len(nonzero):iw] = nonzero
        elif direction == "left":
            for r in range(ih):
                row = ig[r, :iw]
                nonzero = row[row != 0]
                expected[r, :len(nonzero)] = nonzero
        if not np.array_equal(expected[:ih, :iw], og[:ih, :iw]):
            return False
    return True


def make_gravity_onnx(direction, pairs):
    sample = np.array(pairs[0]["input"])
    ih, iw = sample.shape
    const_outputs = []
    for p in pairs:
        ig = np.array(p["input"])
        expected = np.zeros((ih, iw), dtype=np.int32)
        if direction == "down":
            for c in range(iw):
                col = ig[:ih, c]
                nonzero = col[col != 0]
                expected[ih - len(nonzero):ih, c] = nonzero
        elif direction == "up":
            for c in range(iw):
                col = ig[:ih, c]
                nonzero = col[col != 0]
                expected[:len(nonzero), c] = nonzero
        elif direction == "right":
            for r in range(ih):
                row = ig[r, :iw]
                nonzero = row[row != 0]
                expected[r, iw - len(nonzero):iw] = nonzero
        elif direction == "left":
            for r in range(ih):
                row = ig[r, :iw]
                nonzero = row[row != 0]
                expected[r, :len(nonzero)] = nonzero
    return None


def detect_border_fill(pairs):
    for p in pairs:
        ig, og = np.array(p["input"]), np.array(p["output"])
        if ig.shape != og.shape:
            return None, None
        ih, iw = ig.shape
        border_colors = set()
        interior = ig[1:-1, 1:-1] if ih > 2 and iw > 2 else ig
        for r in range(ih):
            for c in range(iw):
                if r == 0 or r == ih - 1 or c == 0 or c == iw - 1:
                    if ig[r, c] != 0:
                        border_colors.add(ig[r, c])
        if len(border_colors) != 1:
            return None, None
        fill_color = list(border_colors)[0]
        expected = ig.copy()
        for r in range(1, ih - 1):
            for c in range(1, iw - 1):
                if ig[r, c] == 0:
                    expected[r, c] = fill_color
        if np.array_equal(expected, og):
            return fill_color, "interior"
    return None, None


def detect_outline(pairs):
    for p in pairs:
        ig, og = np.array(p["input"]), np.array(p["output"])
        if ig.shape != og.shape:
            return None
        ih, iw = ig.shape
        expected = np.zeros_like(ig)
        unique_colors = set(ig.flatten()) - {0}
        for color in unique_colors:
            mask = (ig == color)
            labeled, num = scipy_label(mask)
            for lbl in range(1, num + 1):
                comp = labeled == lbl
                for r in range(ih):
                    for c in range(iw):
                        if comp[r, c]:
                            is_border = (r == 0 or r == ih - 1 or c == 0 or c == iw - 1 or
                                         not comp[r - 1, c] or not comp[r + 1, c] or
                                         not comp[r, c - 1] or not comp[r, c + 1])
                            if is_border:
                                expected[r, c] = color
        if np.array_equal(expected, og):
            return "outline"
    return None


def detect_connected_components_color(pairs):
    for p in pairs:
        ig, og = np.array(p["input"]), np.array(p["output"])
        if ig.shape != og.shape:
            return None
        ih, iw = ig.shape
        colored = set(ig.flatten()) - {0}
        if len(colored) < 2:
            return None
        for bg_color in colored:
            expected = ig.copy()
            for color in colored:
                if color == bg_color:
                    continue
                mask = (ig == color)
                labeled, num = scipy_label(mask)
                for lbl in range(1, num + 1):
                    comp_size = np.sum(labeled == lbl)
            if np.array_equal(expected, og):
                return bg_color
    return None


def detect_mirror_completion(pairs):
    for p in pairs:
        ig, og = np.array(p["input"]), np.array(p["output"])
        if ig.shape != og.shape:
            return None
        ih, iw = ig.shape
        mid_h = ih // 2
        mid_w = iw // 2
        for axis in ["h", "v", "both"]:
            expected = ig.copy()
            if axis == "h":
                for r in range(mid_h):
                    mirror_r = ih - 1 - r
                    for c in range(iw):
                        if ig[r, c] == 0 and ig[mirror_r, c] != 0:
                            expected[r, c] = ig[mirror_r, c]
                        elif ig[mirror_r, c] == 0 and ig[r, c] != 0:
                            expected[mirror_r, c] = ig[r, c]
            elif axis == "v":
                for c in range(mid_w):
                    mirror_c = iw - 1 - c
                    for r in range(ih):
                        if ig[r, c] == 0 and ig[r, mirror_c] != 0:
                            expected[r, c] = ig[r, mirror_c]
                        elif ig[r, mirror_c] == 0 and ig[r, c] != 0:
                            expected[r, mirror_c] = ig[r, c]
            if np.array_equal(expected, og):
                return axis
    return None


def detect_diagonal_flip(pairs):
    for axis in ["main", "anti"]:
        valid = True
        for p in pairs:
            ig, og = np.array(p["input"]), np.array(p["output"])
            if axis == "main":
                expected = ig.T
            else:
                expected = np.fliplr(np.flipud(ig.T))
            if expected.shape != og.shape or not np.array_equal(expected, og):
                valid = False
                break
        if valid:
            return axis
    return None


def detect_invert_colors(pairs):
    for p in pairs:
        ig, og = np.array(p["input"]), np.array(p["output"])
        if ig.shape != og.shape:
            return False
        colors_in = set(ig.flatten())
        colors_out = set(og.flatten())
        if colors_in != colors_out:
            return False
        for r in range(ig.shape[0]):
            for c in range(ig.shape[1]):
                if ig[r, c] == 0 and og[r, c] != 0:
                    pass
                elif ig[r, c] != 0 and og[r, c] == 0:
                    pass
                elif ig[r, c] != og[r, c]:
                    return False
    return False


def detect_color_swap_two(pairs):
    for p in pairs:
        ig, og = np.array(p["input"]), np.array(p["output"])
        if ig.shape != og.shape:
            return None
    all_colors = set()
    for p in pairs:
        all_colors |= set(np.array(p["input"]).flatten())
        all_colors |= set(np.array(p["output"]).flatten())
    all_colors.discard(0)
    colors = list(all_colors)
    for c1, c2 in itertools.combinations(colors, 2):
        valid = True
        for p in pairs:
            ig, og = np.array(p["input"]), np.array(p["output"])
            exp = ig.copy()
            exp[ig == c1] = c2
            exp[ig == c2] = c1
            if not np.array_equal(exp, og):
                valid = False
                break
        if valid:
            return {c1: c2, c2: c1}
    return None


def detect_rotation_and_color(pairs):
    for angle in [90, 180, 270]:
        k = angle // 90
        color_maps = []
        valid = True
        for p in pairs:
            ig, og = np.array(p["input"]), np.array(p["output"])
            rotated = np.rot90(ig, k)
            if rotated.shape != og.shape:
                valid = False
                break
            cmap = {}
            for a, b in zip(rotated.flatten(), og.flatten()):
                if a in cmap and cmap[a] != b:
                    valid = False
                    break
                cmap[a] = b
            if not valid:
                break
            color_maps.append(cmap)
        if valid and color_maps:
            common = color_maps[0]
            for cm in color_maps[1:]:
                if cm != common:
                    common = None
                    break
            if common is not None and any(k != v for k, v in common.items()):
                return angle, common
    return None, None


def detect_flip_and_color(pairs):
    for direction in ["vertical", "horizontal", "both"]:
        color_maps = []
        valid = True
        for p in pairs:
            ig, og = np.array(p["input"]), np.array(p["output"])
            if direction == "vertical":
                flipped = np.flipud(ig)
            elif direction == "horizontal":
                flipped = np.fliplr(ig)
            else:
                flipped = np.flipud(np.fliplr(ig))
            if flipped.shape != og.shape:
                valid = False
                break
            cmap = {}
            for a, b in zip(flipped.flatten(), og.flatten()):
                if a in cmap and cmap[a] != b:
                    valid = False
                    break
                cmap[a] = b
            if not valid:
                break
            color_maps.append(cmap)
        if valid and color_maps:
            common = color_maps[0]
            for cm in color_maps[1:]:
                if cm != common:
                    common = None
                    break
            if common is not None and any(k != v for k, v in common.items()):
                return direction, common
    return None, None


def make_color_remap_onnx_from_map(color_map):
    weight = np.zeros((C, C, 1, 1), dtype=np.float32)
    for src in range(C):
        dst = color_map.get(src, src)
        if 0 <= dst < C:
            weight[dst, src, 0, 0] = 1.0
    return weight


def make_rotation_color_onnx(angle, color_map):
    k = angle // 90
    if k == 1:
        gi_fn = lambda r, c: (c, H - 1 - r)
    elif k == 2:
        gi_fn = lambda r, c: (H - 1 - r, W - 1 - c)
    elif k == 3:
        gi_fn = lambda r, c: (W - 1 - c, r)
    else:
        gi_fn = lambda r, c: (r, c)

    gather_indices = gather_indices_for_transform(gi_fn)
    color_weight = make_color_remap_onnx_from_map(color_map)

    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])

    shape_chw = onh.from_array(np.array([1, C, H * W], dtype=np.int64), name="shape_chw")
    shape_chw_node = oh.make_node("Constant", [], ["shape_chw"], value=shape_chw)
    inp_reshape = oh.make_node("Reshape", ["input", "shape_chw"], ["inp_chw"])
    gi_tensor = onh.from_array(gather_indices.astype(np.int64), name="gather_idx")
    gi_node = oh.make_node("Constant", [], ["gather_idx"], value=gi_tensor)
    gather_node = oh.make_node("Gather", ["inp_chw", "gather_idx"], ["gathered"], axis=2)
    out_shape = onh.from_array(np.array([1, C, H, W], dtype=np.int64), name="out_shape")
    out_shape_node = oh.make_node("Constant", [], ["out_shape"], value=out_shape)
    reshape_back = oh.make_node("Reshape", ["gathered", "out_shape"], ["rotated"])

    w_tensor = onh.from_array(color_weight, name="conv_weight")
    w_node = oh.make_node("Constant", [], ["conv_weight"], value=w_tensor)
    conv_node = oh.make_node("Conv", ["rotated", "conv_weight"], ["output"],
                             kernel_shape=[1, 1], pads=[0, 0, 0, 0])

    nodes = [shape_chw_node, inp_reshape, gi_node, gather_node,
             out_shape_node, reshape_back, w_node, conv_node]
    graph = oh.make_graph(nodes, "rot_color_net", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model


def make_flip_color_onnx(direction, color_map):
    if direction == "vertical":
        fn = lambda r, c: (H - 1 - r, c)
    elif direction == "horizontal":
        fn = lambda r, c: (r, W - 1 - c)
    else:
        fn = lambda r, c: (H - 1 - r, W - 1 - c)

    gather_indices = gather_indices_for_transform(fn)
    color_weight = make_color_remap_onnx_from_map(color_map)

    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])

    shape_chw = onh.from_array(np.array([1, C, H * W], dtype=np.int64), name="shape_chw")
    shape_chw_node = oh.make_node("Constant", [], ["shape_chw"], value=shape_chw)
    inp_reshape = oh.make_node("Reshape", ["input", "shape_chw"], ["inp_chw"])
    gi_tensor = onh.from_array(gather_indices.astype(np.int64), name="gather_idx")
    gi_node = oh.make_node("Constant", [], ["gather_idx"], value=gi_tensor)
    gather_node = oh.make_node("Gather", ["inp_chw", "gather_idx"], ["gathered"], axis=2)
    out_shape = onh.from_array(np.array([1, C, H, W], dtype=np.int64), name="out_shape")
    out_shape_node = oh.make_node("Constant", [], ["out_shape"], value=out_shape)
    reshape_back = oh.make_node("Reshape", ["gathered", "out_shape"], ["flipped"])

    w_tensor = onh.from_array(color_weight, name="conv_weight")
    w_node = oh.make_node("Constant", [], ["conv_weight"], value=w_tensor)
    conv_node = oh.make_node("Conv", ["flipped", "conv_weight"], ["output"],
                             kernel_shape=[1, 1], pads=[0, 0, 0, 0])

    nodes = [shape_chw_node, inp_reshape, gi_node, gather_node,
             out_shape_node, reshape_back, w_node, conv_node]
    graph = oh.make_graph(nodes, "flip_color_net", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model


def detect_checkerboard_pattern(pairs):
    for p in pairs:
        ig, og = np.array(p["input"]), np.array(p["output"])
        if ig.shape != og.shape:
            return None
        ih, iw = ig.shape
        colors = list(set(og.flatten()) - {0})
        if len(colors) != 2:
            return None
        c1, c2 = colors[0], colors[1]
        for offset in [0, 1]:
            valid = True
            for r in range(ih):
                for c in range(iw):
                    expected_color = c1 if (r + c + offset) % 2 == 0 else c2
                    if og[r, c] != expected_color:
                        valid = False
                        break
                if not valid:
                    break
            if valid:
                return c1, c2, offset
    return None


def detect_translation(pairs):
    results = []
    for p in pairs:
        ig, og = np.array(p["input"]), np.array(p["output"])
        if ig.shape != og.shape:
            return None
        ih, iw = ig.shape
        found = False
        for dr in range(-ih + 1, ih):
            for dc in range(-iw + 1, iw):
                if dr == 0 and dc == 0:
                    continue
                shifted = np.zeros_like(ig)
                for r in range(ih):
                    for c in range(iw):
                        nr, nc = r + dr, c + dc
                        if 0 <= nr < ih and 0 <= nc < iw:
                            shifted[nr, nc] = ig[r, c]
                if np.array_equal(shifted, og):
                    results.append((dr, dc))
                    found = True
                    break
            if found:
                break
        if not found:
            return None
    if len(set(results)) == 1:
        return results[0]
    return None


def make_translation_onnx(dr, dc):
    fn = lambda r, c: (r - dr, c - dc)
    gi = np.zeros(H * W, dtype=np.int64)
    for r in range(H):
        for c in range(W):
            nr, nc = r - dr, c - dc
            if 0 <= nr < H and 0 <= nc < W:
                gi[r * W + c] = nr * W + nc
            else:
                gi[r * W + c] = 0
    return make_onnx_from_gather(gi, "translation_net")


def detect_quarter_tile(pairs):
    for p in pairs:
        ig, og = np.array(p["input"]), np.array(p["output"])
        ih, iw = ig.shape
        oh_, ow_ = og.shape
        if oh_ != 2 * ih or ow_ != 2 * iw:
            return None
        top_left = og[:ih, :iw]
        top_right = og[:ih, iw:]
        bot_left = og[ih:, :iw]
        bot_right = og[ih:, iw:]
        variants = {
            "tl": ig,
            "tr": np.fliplr(ig),
            "bl": np.flipud(ig),
            "br": np.rot90(ig, 2),
        }
        if not (np.array_equal(top_left, ig) and np.array_equal(top_right, np.fliplr(ig)) and
                np.array_equal(bot_left, np.flipud(ig)) and np.array_equal(bot_right, np.rot90(ig, 2))):
            return None
    return "mirror_quad"


def detect_object_count_to_size(pairs):
    for p in pairs:
        ig, og = np.array(p["input"]), np.array(p["output"])
        pass
    return None


def detect_selective_color_preserve(pairs):
    for p in pairs:
        ig, og = np.array(p["input"]), np.array(p["output"])
        if ig.shape != og.shape:
            return None
        ih, iw = ig.shape
        colors_in = set(ig.flatten()) - {0}
        colors_out = set(og.flatten()) - {0}
        if not colors_out.issubset(colors_in):
            return None
        removed = colors_in - colors_out
        if not removed:
            return None
        expected = ig.copy()
        for color in removed:
            expected[expected == color] = 0
        if np.array_equal(expected, og):
            return removed
    return None


def detect_fill_holes(pairs):
    for p in pairs:
        ig, og = np.array(p["input"]), np.array(p["output"])
        if ig.shape != og.shape:
            return None
        ih, iw = ig.shape
        nz_in = np.sum(ig != 0)
        nz_out = np.sum(og != 0)
        if nz_out < nz_in:
            return None
        colored_in = set(ig.flatten()) - {0}
        for color in colored_in:
            mask = (ig == color).astype(np.uint8)
            labeled, num = scipy_label(mask)
            for lbl in range(1, num + 1):
                comp = labeled == lbl
                r_min, r_max = np.where(comp.any(axis=1))[0][[0, -1]]
                c_min, c_max = np.where(comp.any(axis=0))[0][[0, -1]]
                region_in = ig[r_min:r_max + 1, c_min:c_max + 1]
                region_out = og[r_min:r_max + 1, c_min:c_max + 1]
                interior = np.zeros_like(region_in)
                for r in range(1, region_in.shape[0] - 1):
                    for c in range(1, region_in.shape[1] - 1):
                        if region_in[r, c] == 0:
                            interior[r, c] = color
    return None


def detect_majority_color(pairs):
    for p in pairs:
        ig, og = np.array(p["input"]), np.array(p["output"])
        if ig.shape != og.shape:
            return None


def detect_frame_extraction(pairs):
    results = []
    for p in pairs:
        ig, og = np.array(p["input"]), np.array(p["output"])
        ih, iw = ig.shape
        oh_, ow_ = og.shape
        if oh_ >= ih or ow_ >= iw:
            return None
        colors = set(ig.flatten()) - {0}
        for color in colors:
            for r0 in range(ih - oh_ + 1):
                for c0 in range(iw - ow_ + 1):
                    region = ig[r0:r0 + oh_, c0:c0 + ow_]
                    border_matches = True
                    for r in range(oh_):
                        for c in range(ow_):
                            is_border = (r == 0 or r == oh_ - 1 or c == 0 or c == ow_ - 1)
                            if is_border and region[r, c] != color:
                                border_matches = False
                    if border_matches and np.array_equal(region, og):
                        results.append((r0, c0, oh_, ow_))
    if results:
        r0, c0, oh_, ow_ = results[0]
        consistent = all(r == results[0] for r in results)
        if consistent:
            return r0, c0, oh_, ow_
    return None


def try_learned_conv_torch(pairs, kernel_size=1, steps=1000):
    try:
        import torch
        import torch.nn as nn
    except ImportError:
        return None

    if not all(np.array(p["input"]).shape == np.array(p["output"]).shape for p in pairs):
        return None

    inp_t = torch.tensor(np.stack([grid_to_tensor(p["input"])[0] for p in pairs]), dtype=torch.float32)
    out_t = torch.tensor(np.stack([grid_to_tensor(p["output"])[0] for p in pairs]), dtype=torch.float32)

    pad = kernel_size // 2
    conv = nn.Conv2d(C, C, kernel_size=kernel_size, padding=pad, bias=False)
    nn.init.zeros_(conv.weight)

    optimizer = torch.optim.Adam(conv.parameters(), lr=0.05)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=50, factor=0.5)
    best_loss = float("inf")
    best_state = None

    for step in range(steps):
        optimizer.zero_grad()
        pred = conv(inp_t)
        loss = nn.functional.mse_loss(pred, out_t)
        loss.backward()
        optimizer.step()
        scheduler.step(loss.item())
        if loss.item() < best_loss:
            best_loss = loss.item()
            best_state = copy.deepcopy(conv.state_dict())

    if best_loss > 0.05:
        return None

    conv.load_state_dict(best_state)
    weight_np = conv.weight.detach().numpy()
    return make_conv_onnx(weight_np, kernel_size, f"learned_conv_{kernel_size}")


def try_multi_layer_conv(pairs):
    try:
        import torch
        import torch.nn as nn
    except ImportError:
        return None

    if not all(np.array(p["input"]).shape == np.array(p["output"]).shape for p in pairs):
        return None

    inp_t = torch.tensor(np.stack([grid_to_tensor(p["input"])[0] for p in pairs]), dtype=torch.float32)
    out_t = torch.tensor(np.stack([grid_to_tensor(p["output"])[0] for p in pairs]), dtype=torch.float32)

    class TwoLayerNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv2d(C, 32, kernel_size=3, padding=1, bias=True)
            self.conv2 = nn.Conv2d(32, C, kernel_size=1, bias=True)
            self.relu = nn.ReLU()

        def forward(self, x):
            return self.conv2(self.relu(self.conv1(x)))

    net = TwoLayerNet()
    optimizer = torch.optim.Adam(net.parameters(), lr=0.01)
    best_loss = float("inf")
    best_state = None

    for step in range(2000):
        optimizer.zero_grad()
        pred = net(inp_t)
        loss = nn.functional.mse_loss(pred, out_t)
        loss.backward()
        optimizer.step()
        if loss.item() < best_loss:
            best_loss = loss.item()
            best_state = copy.deepcopy(net.state_dict())
        if best_loss < 0.01:
            break

    if best_loss > 0.05:
        return None

    net.load_state_dict(best_state)

    w1 = net.conv1.weight.detach().numpy().astype(np.float32)
    b1 = net.conv1.bias.detach().numpy().astype(np.float32)
    w2 = net.conv2.weight.detach().numpy().astype(np.float32)
    b2 = net.conv2.bias.detach().numpy().astype(np.float32)

    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])

    w1t = onh.from_array(w1, name="w1")
    b1t = onh.from_array(b1, name="b1")
    w2t = onh.from_array(w2, name="w2")
    b2t = onh.from_array(b2, name="b2")

    nodes = [
        oh.make_node("Constant", [], ["w1"], value=w1t),
        oh.make_node("Constant", [], ["b1"], value=b1t),
        oh.make_node("Conv", ["input", "w1", "b1"], ["conv1_out"],
                     kernel_shape=[3, 3], pads=[1, 1, 1, 1]),
        oh.make_node("Relu", ["conv1_out"], ["relu_out"]),
        oh.make_node("Constant", [], ["w2"], value=w2t),
        oh.make_node("Constant", [], ["b2"], value=b2t),
        oh.make_node("Conv", ["relu_out", "w2", "b2"], ["output"],
                     kernel_shape=[1, 1], pads=[0, 0, 0, 0]),
    ]
    graph = oh.make_graph(nodes, "two_layer_net", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model


def try_general_linear(train_pairs, all_pairs):
    if not all(np.array(p["input"]).shape == (H, W) and
               np.array(p["output"]).shape == (H, W) for p in train_pairs):
        return None

    inp_tensors = np.stack([grid_to_tensor(p["input"])[0] for p in train_pairs])
    out_tensors = np.stack([grid_to_tensor(p["output"])[0] for p in train_pairs])

    n, c, h, w = inp_tensors.shape
    X_mat = inp_tensors.reshape(n, -1)
    Y_mat = out_tensors.reshape(n, -1)

    try:
        W_sol, _, _, _ = np.linalg.lstsq(X_mat, Y_mat, rcond=None)
        Y_pred = X_mat @ W_sol
        err = np.max(np.abs(Y_pred.round() - Y_mat))
        if err > 0.1:
            return None
    except Exception:
        return None

    W_np = W_sol.T.astype(np.float32)

    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])

    flat_shape = onh.from_array(np.array([1, C * H * W], dtype=np.int64), name="flat_shape")
    flat_shape_node = oh.make_node("Constant", [], ["flat_shape"], value=flat_shape)
    reshape_in = oh.make_node("Reshape", ["input", "flat_shape"], ["inp_flat"])
    w_tensor = onh.from_array(W_np, name="gemm_weight")
    w_node = oh.make_node("Constant", [], ["gemm_weight"], value=w_tensor)
    gemm_node = oh.make_node("Gemm", ["inp_flat", "gemm_weight"], ["out_flat"],
                             transB=0, alpha=1.0, beta=0.0)
    out_shape = onh.from_array(np.array([1, C, H, W], dtype=np.int64), name="out_shape")
    out_shape_node = oh.make_node("Constant", [], ["out_shape"], value=out_shape)
    reshape_out = oh.make_node("Reshape", ["out_flat", "out_shape"], ["output"])

    nodes = [flat_shape_node, reshape_in, w_node, gemm_node, out_shape_node, reshape_out]
    graph = oh.make_graph(nodes, "linear_net", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8

    if check_model_correct(model, all_pairs):
        return model
    return None


def detect_same_size(pairs):
    return all(np.array(p["input"]).shape == np.array(p["output"]).shape for p in pairs)


def detect_identity(pairs):
    return all(np.array_equal(np.array(p["input"]), np.array(p["output"])) for p in pairs)


def detect_const_output(pairs):
    outputs = [str(p["output"]) for p in pairs]
    return len(set(outputs)) == 1


def make_output_from_input_reorder(pairs, reorder_fn):
    gather_indices = np.zeros(H * W, dtype=np.int64)
    for r in range(H):
        for c in range(W):
            nr, nc = reorder_fn(r, c)
            nr = max(0, min(H - 1, nr))
            nc = max(0, min(W - 1, nc))
            gather_indices[r * W + c] = nr * W + nc
    return gather_indices


def detect_anti_diagonal_flip(pairs):
    for p in pairs:
        ig, og = np.array(p["input"]), np.array(p["output"])
        n = min(ig.shape[0], ig.shape[1])
        if ig.shape != og.shape:
            return False
        for r in range(ig.shape[0]):
            for c in range(ig.shape[1]):
                nr, nc = ig.shape[1] - 1 - c, ig.shape[0] - 1 - r
                if nr < og.shape[0] and nc < og.shape[1]:
                    if ig[r, c] != og[nr, nc]:
                        return False
    return True


def detect_scale_down(pairs):
    for factor in [2, 3, 4]:
        valid = True
        for p in pairs:
            ig, og = np.array(p["input"]), np.array(p["output"])
            ih, iw = ig.shape
            oh_, ow_ = og.shape
            if ih != oh_ * factor or iw != ow_ * factor:
                valid = False
                break
            downscaled = ig[::factor, ::factor]
            if downscaled.shape != og.shape or not np.array_equal(downscaled, og):
                valid = False
                break
        if valid:
            return factor
    return None


def make_scale_down_onnx(factor):
    gi = np.zeros(H * W, dtype=np.int64)
    mask = np.zeros((1, C, H, W), dtype=np.float32)
    oh_ = H // factor
    ow_ = W // factor
    for r in range(H):
        for c in range(W):
            if r < oh_ and c < ow_:
                src_r = r * factor
                src_c = c * factor
                gi[r * W + c] = src_r * W + src_c
                mask[0, :, r, c] = 1.0
    if oh_ < H or ow_ < W:
        return make_masked_gather_onnx(gi, mask, "scale_down_net")
    return make_onnx_from_gather(gi, "scale_down_net")


def detect_border_removal(pairs):
    results = []
    for p in pairs:
        ig, og = np.array(p["input"]), np.array(p["output"])
        ih, iw = ig.shape
        oh_, ow_ = og.shape
        if oh_ == ih - 2 and ow_ == iw - 2:
            if np.array_equal(ig[1:-1, 1:-1], og):
                results.append(True)
            else:
                return None
        else:
            return None
    return True if results else None


def make_border_removal_onnx():
    return make_crop_onnx(1, 1, H - 2, W - 2)


def detect_upscale_with_pattern(pairs):
    for factor in [2, 3]:
        valid = True
        for p in pairs:
            ig, og = np.array(p["input"]), np.array(p["output"])
            ih, iw = ig.shape
            oh_, ow_ = og.shape
            if oh_ != ih * factor or ow_ != iw * factor:
                valid = False
                break
        if valid:
            return factor
    return None


def detect_row_col_swap(pairs):
    for axis in ["rows", "cols"]:
        for p in pairs:
            ig, og = np.array(p["input"]), np.array(p["output"])
            if ig.shape != og.shape:
                return None
            ih, iw = ig.shape
            if axis == "rows":
                perm = None
                for i in range(ih):
                    for j in range(ih):
                        if np.array_equal(ig[i, :], og[j, :]):
                            if perm is None:
                                perm = {}
                            perm[j] = i
                            break
                if perm and len(perm) == ih:
                    return axis, perm
    return None


def detect_color_to_shape(pairs):
    color_maps = []
    for p in pairs:
        ig, og = np.array(p["input"]), np.array(p["output"])
        if ig.shape != og.shape:
            return None
        cmap = {}
        for a, b in zip(ig.flatten(), og.flatten()):
            if a in cmap and cmap[a] != b:
                return None
            cmap[a] = b
        color_maps.append(cmap)
    if not color_maps:
        return None
    common = color_maps[0]
    for cm in color_maps[1:]:
        if cm != common:
            return None
    return common


def detect_90deg_variants_with_crop(pairs):
    all_in = [np.array(p["input"]) for p in pairs]
    all_out = [np.array(p["output"]) for p in pairs]
    ih, iw = all_in[0].shape
    oh_, ow_ = all_out[0].shape

    if oh_ > ih and ow_ > iw:
        return None

    for angle in [90, 180, 270]:
        k = angle // 90
        crop_results = []
        valid = True
        for ig, og in zip(all_in, all_out):
            rotated = np.rot90(ig, k)
            rh, rw = rotated.shape
            toh, tow = og.shape
            found = False
            for r0 in range(rh - toh + 1):
                for c0 in range(rw - tow + 1):
                    if np.array_equal(rotated[r0:r0 + toh, c0:c0 + tow], og):
                        crop_results.append((angle, r0, c0))
                        found = True
                        break
                if found:
                    break
            if not found:
                valid = False
                break
        if valid and len(set(crop_results)) == 1:
            return crop_results[0]
    return None


def solve_task(task_data, task_name=""):
    train_pairs = task_data.get("train", [])
    test_pairs = task_data.get("test", [])
    arc_gen = task_data.get("arc-gen", [])
    if not train_pairs:
        return make_identity_onnx()

    arc_gen_sample = arc_gen[:min(10, len(arc_gen))]
    all_pairs = train_pairs + test_pairs + arc_gen_sample

    if detect_identity(train_pairs):
        model = make_identity_onnx()
        if check_model_correct(model, all_pairs):
            return model

    color_map = detect_color_to_shape(train_pairs)
    if color_map:
        model = make_color_remap_onnx(color_map)
        if check_model_correct(model, all_pairs):
            return model

    color_swap = detect_color_swap_two(train_pairs)
    if color_swap:
        model = make_color_remap_onnx(color_swap)
        if check_model_correct(model, all_pairs):
            return model

    rot = detect_rotation(train_pairs)
    if rot is not None:
        model = make_rotation_onnx(rot)
        if check_model_correct(model, all_pairs):
            return model

    flip = detect_flip(train_pairs)
    if flip is not None:
        model = make_flip_onnx(flip)
        if check_model_correct(model, all_pairs):
            return model

    if detect_transpose(train_pairs):
        model = make_transpose_onnx()
        if check_model_correct(model, all_pairs):
            return model

    if detect_anti_diagonal_flip(train_pairs):
        fn = lambda r, c: (W - 1 - c, H - 1 - r)
        gi = gather_indices_for_transform(fn)
        model = make_onnx_from_gather(gi, "anti_diag_net")
        if check_model_correct(model, all_pairs):
            return model

    rot_angle, rot_color = detect_rotation_and_color(train_pairs)
    if rot_angle is not None and rot_color is not None:
        model = make_rotation_color_onnx(rot_angle, rot_color)
        if check_model_correct(model, all_pairs):
            return model

    flip_dir, flip_color = detect_flip_and_color(train_pairs)
    if flip_dir is not None and flip_color is not None:
        model = make_flip_color_onnx(flip_dir, flip_color)
        if check_model_correct(model, all_pairs):
            return model

    scale = detect_scale(train_pairs)
    if scale is not None:
        model = make_scale_onnx(scale)
        if check_model_correct(model, all_pairs):
            return model

    scale_down = detect_scale_down(train_pairs)
    if scale_down is not None:
        model = make_scale_down_onnx(scale_down)
        if check_model_correct(model, all_pairs):
            return model

    tile = detect_tile(train_pairs)
    if tile is not None:
        model = make_tile_onnx(*tile)
        if check_model_correct(model, all_pairs):
            return model

    crop = detect_crop(train_pairs)
    if crop is not None:
        r0, c0, oh_, ow_ = crop
        model = make_crop_onnx(r0, c0, oh_, ow_)
        if check_model_correct(model, all_pairs):
            return model

    if detect_border_removal(train_pairs):
        model = make_border_removal_onnx()
        if check_model_correct(model, all_pairs):
            return model

    for dr in range(-5, 6):
        for dc in range(-5, 6):
            if dr == 0 and dc == 0:
                continue
            valid = True
            for p in train_pairs:
                ig, og = np.array(p["input"]), np.array(p["output"])
                if ig.shape != og.shape:
                    valid = False
                    break
                ih, iw = ig.shape
                shifted = np.zeros_like(ig)
                for r in range(ih):
                    for c in range(iw):
                        nr, nc = r + dr, c + dc
                        if 0 <= nr < ih and 0 <= nc < iw:
                            shifted[nr, nc] = ig[r, c]
                if not np.array_equal(shifted, og):
                    valid = False
                    break
            if valid:
                model = make_translation_onnx(dr, dc)
                if check_model_correct(model, all_pairs):
                    return model

    perm = detect_pixel_permutation(train_pairs)
    if perm is not None:
        model = make_onnx_from_gather(perm, "pixel_perm")
        if check_model_correct(model, all_pairs):
            return model

    for ks in [1, 3, 5]:
        model = try_learned_conv_torch(train_pairs, kernel_size=ks, steps=800)
        if model is not None and check_model_correct(model, all_pairs):
            return model

    model = try_multi_layer_conv(train_pairs)
    if model is not None and check_model_correct(model, all_pairs):
        return model

    model = try_general_linear(train_pairs, all_pairs)
    if model is not None:
        return model

    removed = detect_selective_color_preserve(train_pairs)
    if removed is not None:
        preserved_colors = set(range(C)) - removed
        cmap = {c: c for c in preserved_colors}
        for c in removed:
            cmap[c] = 0
        model = make_color_remap_onnx(cmap)
        if check_model_correct(model, all_pairs):
            return model

    for d in ["up", "down", "left", "right"]:
        if detect_gravity(train_pairs, d):
            pass

    if detect_const_output(train_pairs):
        const_out = grid_to_tensor(train_pairs[0]["output"])
        model = make_const_onnx(const_out)
        if check_model_correct(model, all_pairs):
            return model

    return None


def save_model(model, path):
    try:
        onnx.checker.check_model(model)
    except Exception:
        pass
    with open(path, "wb") as f:
        f.write(model.SerializeToString())


In [ ]:
%%writefile fast_solver.py
"""
NeuroGolf 2026 — cost-optimal ONNX solver for ARC-AGI tasks
=============================================================

Scoring: max(1, 25 - ln(cost)) per task, cost = params + bytes + MACs.

Key ideas vs. the starter code:

1.  Rotations/flips use Transpose + Slice (0 params, 0 MACs) instead of a
    900-element int64 gather table (7200 bytes). Same-canvas variants for
    30x30 inputs, and "framed" variants (Slice content -> transform -> Pad)
    for any fixed non-30x30 size.

2.  Color remaps use Gather on the channel axis: 10 int64 params, 0 MACs.
    The starter's 1x1 Conv cost 100 params + 90 000 MACs for the same thing.

3.  Constant outputs emit Constant + Identity — no Mul(x,0) + Add chain
    (which wasted 9 000 MACs in the starter).

4.  Crop = Slice + Pad.  Translation = Slice + Pad.  Zero params, zero MACs.

5.  Every candidate is measured via cost(model) and the cheapest correct one
    wins — we don't take the first correct one.

6.  Validation uses train + test + up to 10 arc-gen samples so the model
    generalises to the private benchmark.

Running this in a Kaggle notebook:
  * Place the 400 task*.json files under the input path (default tries several
    common Kaggle mount points).
  * Run the cell. ONNX files are emitted to /kaggle/working and bundled into
    submission.zip at the end.
"""

import os, json, math, zipfile, copy, itertools, sys
from pathlib import Path
from typing import Optional, Tuple, List, Dict, Any, Iterable

import numpy as np

# Robust onnx/onnxruntime bootstrap (Kaggle kernels usually have both) ---------
try:
    import onnx
    import onnx.helper as oh
    import onnx.numpy_helper as onh
    from onnx import TensorProto
except ImportError:
    os.system("pip install onnx -q")
    import onnx
    import onnx.helper as oh
    import onnx.numpy_helper as onh
    from onnx import TensorProto

try:
    import onnxruntime as ort
except ImportError:
    os.system("pip install onnxruntime -q")
    import onnxruntime as ort


# =============================================================================
# Constants
# =============================================================================

C, H, W = 10, 30, 30
MAX_FILE_BYTES = int(1.44 * 1024 * 1024)
DISALLOWED_OPS = {"Loop", "Scan", "NonZero", "Unique", "Script", "Function"}
OPSET = [oh.make_opsetid("", 17)]
IR_VERSION = 8


# =============================================================================
# Tensor <-> grid
# =============================================================================

def grid_to_tensor(grid) -> np.ndarray:
    g = np.asarray(grid, dtype=np.int32)
    h, w = g.shape
    t = np.zeros((1, C, H, W), dtype=np.float32)
    for r in range(h):
        for c in range(w):
            v = int(g[r, c])
            if 0 <= v < C:
                t[0, v, r, c] = 1.0
    return t

def tensor_to_grid(t: np.ndarray, out_h: int, out_w: int) -> List[List[int]]:
    arr = t[0, :, :out_h, :out_w]
    grid: List[List[int]] = []
    for r in range(out_h):
        row = []
        for c in range(out_w):
            col_vals = arr[:, r, c]
            if col_vals.max() < 0.5:
                row.append(0)
            else:
                row.append(int(np.argmax(col_vals)))
        grid.append(row)
    return grid


# =============================================================================
# Graph helpers
# =============================================================================

def _X():
    return oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])

def _Y():
    return oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])

def _const(name: str, arr: np.ndarray):
    t = onh.from_array(arr, name=name + "_v")
    return oh.make_node("Constant", [], [name], value=t)

def _make_model(nodes, graph_name="net"):
    g = oh.make_graph(nodes, graph_name, [_X()], [_Y()])
    m = oh.make_model(g, opset_imports=OPSET)
    m.ir_version = IR_VERSION
    return m


# =============================================================================
# Cost / validation
# =============================================================================

def model_bytes(model) -> bytes:
    return model.SerializeToString()

def check_fits(model) -> bool:
    return len(model_bytes(model)) <= MAX_FILE_BYTES

def check_ops_allowed(model) -> bool:
    for node in model.graph.node:
        if node.op_type in DISALLOWED_OPS:
            return False
    return True

def _iter_constants(model):
    for init in model.graph.initializer:
        yield onh.to_array(init)
    for node in model.graph.node:
        if node.op_type == "Constant":
            for attr in node.attribute:
                if attr.name == "value":
                    yield onh.to_array(attr.t)

def count_params(model) -> int:
    return sum(int(a.size) for a in _iter_constants(model))

def _shape_map(model):
    shapes: Dict[str, Tuple[int, ...]] = {}
    for vi in list(model.graph.input) + list(model.graph.output):
        dims = tuple(d.dim_value if d.dim_value > 0 else 1
                     for d in vi.type.tensor_type.shape.dim)
        shapes[vi.name] = dims
    for init in model.graph.initializer:
        shapes[init.name] = tuple(init.dims)
    for node in model.graph.node:
        if node.op_type == "Constant":
            for attr in node.attribute:
                if attr.name == "value":
                    shapes[node.output[0]] = tuple(attr.t.dims) if attr.t.dims else ()
    return shapes

def estimate_macs(model) -> int:
    """Conservative MAC estimate.

    Organisers count "multiply-accumulate operations required to execute the
    network". For Conv/Gemm/MatMul this is standard. We additionally count
    element-wise Mul's output element count as a safety margin — a scalar
    mul+add doesn't accumulate across inputs but we'd rather over-count than
    under-count when picking the cheapest candidate."""
    macs = 0
    shapes = _shape_map(model)
    for node in model.graph.node:
        if node.op_type == "Conv":
            w = shapes.get(node.input[1]); x = shapes.get(node.input[0])
            if w and x and len(w) == 4 and len(x) == 4:
                out_c, in_c, kh, kw = w
                _, _, xh, xw = x
                macs += int(out_c * in_c * kh * kw * xh * xw)
        elif node.op_type == "Gemm":
            a = shapes.get(node.input[0]); w = shapes.get(node.input[1])
            if a and w and len(a) >= 2 and len(w) >= 2:
                m = int(a[0]); k = int(a[1]); n = int(w[0] if w[0] != k else w[1])
                macs += m * k * n
        elif node.op_type == "MatMul":
            a = shapes.get(node.input[0]); b = shapes.get(node.input[1])
            if a and b and len(a) >= 2 and len(b) >= 2:
                macs += int(a[-2]) * int(a[-1]) * int(b[-1])
    return macs

def cost(model) -> int:
    return count_params(model) + len(model_bytes(model)) + estimate_macs(model)


def check_model_correct(model, pairs) -> bool:
    if not pairs:
        return False
    if not check_fits(model):
        return False
    if not check_ops_allowed(model):
        return False
    try:
        so = ort.SessionOptions()
        so.log_severity_level = 3
        sess = ort.InferenceSession(model_bytes(model), sess_options=so,
                                    providers=["CPUExecutionProvider"])
    except Exception:
        return False
    try:
        for p in pairs:
            inp = grid_to_tensor(p["input"])
            og = np.asarray(p["output"])
            oh_, ow_ = og.shape
            pred = sess.run(None, {"input": inp})[0]
            outside = pred.copy()
            outside[0, :, :oh_, :ow_] = 0.0
            if np.max(np.abs(outside)) > 1e-4:
                return False
            if tensor_to_grid(pred, oh_, ow_) != p["output"]:
                return False
        return True
    except Exception:
        return False


# =============================================================================
# Model constructions (ordered by typical cost)
# =============================================================================

# --- Identity ----------------------------------------------------------------

def make_identity():
    return _make_model([oh.make_node("Identity", ["input"], ["output"])], "identity")


# --- Pure constant output ----------------------------------------------------

def make_const(output_grid: np.ndarray):
    og = np.asarray(output_grid, dtype=np.int32)
    oh_, ow_ = og.shape
    padded = np.zeros((1, C, H, W), dtype=np.float32)
    for r in range(oh_):
        for c in range(ow_):
            v = int(og[r, c])
            if 0 <= v < C:
                padded[0, v, r, c] = 1.0
    nodes = [_const("K", padded), oh.make_node("Identity", ["K"], ["output"])]
    return _make_model(nodes, "const_net")


# --- Full-canvas Transpose / Slice-reverse (0 params, 0 MACs) ----------------

def make_transpose_hw():
    return _make_model(
        [oh.make_node("Transpose", ["input"], ["output"], perm=[0, 1, 3, 2])],
        "transpose_hw")

def _slice_reverse_nodes(axes, dims, in_name, out_name, suf=""):
    starts = [d - 1 for d in dims]
    ends   = [-d - 1 for d in dims]
    steps  = [-1] * len(dims)
    return [
        _const(f"rs_s{suf}", np.asarray(starts, dtype=np.int64)),
        _const(f"rs_e{suf}", np.asarray(ends,   dtype=np.int64)),
        _const(f"rs_a{suf}", np.asarray(axes,   dtype=np.int64)),
        _const(f"rs_p{suf}", np.asarray(steps,  dtype=np.int64)),
        oh.make_node("Slice",
                     [in_name, f"rs_s{suf}", f"rs_e{suf}", f"rs_a{suf}", f"rs_p{suf}"],
                     [out_name]),
    ]

def make_flip_h():
    return _make_model(_slice_reverse_nodes([2], [H], "input", "output"), "flip_h")

def make_flip_w():
    return _make_model(_slice_reverse_nodes([3], [W], "input", "output"), "flip_w")

def make_flip_both():
    return _make_model(_slice_reverse_nodes([2, 3], [H, W], "input", "output"), "flip_both")

def make_rot_180():
    return make_flip_both()

def make_rot_90_cw():
    nodes = [oh.make_node("Transpose", ["input"], ["t"], perm=[0, 1, 3, 2])]
    nodes += _slice_reverse_nodes([3], [W], "t", "output")
    return _make_model(nodes, "rot_90_cw")

def make_rot_90_ccw():
    nodes = [oh.make_node("Transpose", ["input"], ["t"], perm=[0, 1, 3, 2])]
    nodes += _slice_reverse_nodes([2], [H], "t", "output")
    return _make_model(nodes, "rot_90_ccw")

def make_anti_transpose():
    nodes = [oh.make_node("Transpose", ["input"], ["t"], perm=[0, 1, 3, 2])]
    nodes += _slice_reverse_nodes([2, 3], [H, W], "t", "output")
    return _make_model(nodes, "anti_transpose")


# --- Framed rotations/flips for fixed (h, w) != 30x30 -----------------------

def _pad_to_canvas(in_name, out_name, out_h, out_w, suf=""):
    pads = np.asarray([0, 0, 0, 0, 0, 0, H - out_h, W - out_w], dtype=np.int64)
    return [
        _const(f"pd{suf}", pads),
        oh.make_node("Pad", [in_name, f"pd{suf}"], [out_name], mode="constant"),
    ]

def _crop_to_content(in_name, out_name, h, w, suf=""):
    return [
        _const(f"cs{suf}", np.asarray([0, 0], dtype=np.int64)),
        _const(f"ce{suf}", np.asarray([h, w], dtype=np.int64)),
        _const(f"ca{suf}", np.asarray([2, 3], dtype=np.int64)),
        oh.make_node("Slice",
                     [in_name, f"cs{suf}", f"ce{suf}", f"ca{suf}"],
                     [out_name]),
    ]

def make_framed_rot180(h, w):
    nodes = _crop_to_content("input", "content", h, w, "0")
    nodes += _slice_reverse_nodes([2, 3], [h, w], "content", "rotated", "r")
    nodes += _pad_to_canvas("rotated", "output", h, w, "p")
    return _make_model(nodes, "framed_rot180")

def make_framed_flip_h(h, w):
    nodes = _crop_to_content("input", "content", h, w, "0")
    nodes += _slice_reverse_nodes([2], [h], "content", "flipped", "r")
    nodes += _pad_to_canvas("flipped", "output", h, w, "p")
    return _make_model(nodes, "framed_flip_h")

def make_framed_flip_w(h, w):
    nodes = _crop_to_content("input", "content", h, w, "0")
    nodes += _slice_reverse_nodes([3], [w], "content", "flipped", "r")
    nodes += _pad_to_canvas("flipped", "output", h, w, "p")
    return _make_model(nodes, "framed_flip_w")

def make_framed_transpose(h, w):
    nodes = _crop_to_content("input", "content", h, w, "0")
    nodes += [oh.make_node("Transpose", ["content"], ["transposed"], perm=[0, 1, 3, 2])]
    nodes += _pad_to_canvas("transposed", "output", w, h, "p")
    return _make_model(nodes, "framed_transpose")

def make_framed_rot_90_cw(h, w):
    nodes = _crop_to_content("input", "content", h, w, "0")
    nodes += [oh.make_node("Transpose", ["content"], ["t"], perm=[0, 1, 3, 2])]
    nodes += _slice_reverse_nodes([3], [h], "t", "rotated", "r")
    nodes += _pad_to_canvas("rotated", "output", w, h, "p")
    return _make_model(nodes, "framed_rot_90_cw")

def make_framed_rot_90_ccw(h, w):
    nodes = _crop_to_content("input", "content", h, w, "0")
    nodes += [oh.make_node("Transpose", ["content"], ["t"], perm=[0, 1, 3, 2])]
    nodes += _slice_reverse_nodes([2], [w], "t", "rotated", "r")
    nodes += _pad_to_canvas("rotated", "output", w, h, "p")
    return _make_model(nodes, "framed_rot_90_ccw")


# --- Color remap via channel-axis Gather ------------------------------------

def make_color_remap_gather(color_map: Dict[int, int]):
    inv: Dict[int, int] = {}
    for s, d in color_map.items():
        inv[d] = s
    idx = np.arange(C, dtype=np.int64)
    mask = np.ones(C, dtype=np.float32)
    src_remapped_away = {s for s, d in color_map.items() if s != d}
    for d in range(C):
        if d in inv:
            idx[d] = inv[d]
        elif d in src_remapped_away:
            idx[d] = 0
            mask[d] = 0.0
        else:
            idx[d] = d
    gi = _const("gi", idx)
    if mask.min() == 1.0:
        gnode = oh.make_node("Gather", ["input", "gi"], ["output"], axis=1)
        return _make_model([gi, gnode], "color_remap")
    gnode = oh.make_node("Gather", ["input", "gi"], ["gathered"], axis=1)
    mk = _const("cm", mask.reshape(1, C, 1, 1))
    mul = oh.make_node("Mul", ["gathered", "cm"], ["output"])
    return _make_model([gi, gnode, mk, mul], "color_remap_masked")


# --- Crop (Slice + Pad) -----------------------------------------------------

def make_crop(r0: int, c0: int, out_h: int, out_w: int):
    nodes = [
        _const("cs", np.asarray([r0, c0], dtype=np.int64)),
        _const("ce", np.asarray([r0 + out_h, c0 + out_w], dtype=np.int64)),
        _const("ca", np.asarray([2, 3], dtype=np.int64)),
        oh.make_node("Slice", ["input", "cs", "ce", "ca"], ["cropped"]),
    ]
    nodes += _pad_to_canvas("cropped", "output", out_h, out_w, "p")
    return _make_model(nodes, "crop_net")


# --- Translation (Slice + Pad with different offsets) -----------------------

def make_translation(dr: int, dc: int):
    """Full-canvas translation. USE ONLY when fixed_size == (H, W), since the
    padded-background region gets shifted and can leak non-zero channel-0
    activations outside the real grid for smaller inputs. For smaller fixed
    sizes, use make_framed_translation."""
    src_r0 = max(0, -dr); src_r1 = min(H, H - dr)
    src_c0 = max(0, -dc); src_c1 = min(W, W - dc)
    dst_r0 = max(0, dr);  dst_c0 = max(0, dc)
    rh = src_r1 - src_r0; rw = src_c1 - src_c0
    if rh <= 0 or rw <= 0:
        return make_const(np.zeros((1, 1), dtype=np.int32))
    nodes = [
        _const("ts", np.asarray([src_r0, src_c0], dtype=np.int64)),
        _const("te", np.asarray([src_r1, src_c1], dtype=np.int64)),
        _const("ta", np.asarray([2, 3], dtype=np.int64)),
        oh.make_node("Slice", ["input", "ts", "te", "ta"], ["sliced"]),
    ]
    pads = np.asarray([
        0, 0, dst_r0, dst_c0,
        0, 0, H - dst_r0 - rh, W - dst_c0 - rw
    ], dtype=np.int64)
    nodes += [
        _const("pp", pads),
        oh.make_node("Pad", ["sliced", "pp"], ["output"], mode="constant"),
    ]
    return _make_model(nodes, "translate_net")


def make_framed_translation(grid_h: int, grid_w: int, dr: int, dc: int):
    """Translate only within the grid's content rectangle [0:grid_h, 0:grid_w],
    leaving the rest of the 30x30 canvas at zero. Used when every example has
    the same grid size (grid_h, grid_w) and content shifts by (dr, dc)."""
    # Source region inside the content rect:
    src_r0 = max(0, -dr); src_r1 = min(grid_h, grid_h - dr)
    src_c0 = max(0, -dc); src_c1 = min(grid_w, grid_w - dc)
    dst_r0 = max(0, dr);  dst_c0 = max(0, dc)
    rh = src_r1 - src_r0; rw = src_c1 - src_c0
    if rh <= 0 or rw <= 0:
        # Everything shifts out of frame -> all-zero output.
        return make_const(np.zeros((1, 1), dtype=np.int32))
    nodes = [
        _const("fts", np.asarray([src_r0, src_c0], dtype=np.int64)),
        _const("fte", np.asarray([src_r1, src_c1], dtype=np.int64)),
        _const("fta", np.asarray([2, 3], dtype=np.int64)),
        oh.make_node("Slice", ["input", "fts", "fte", "fta"], ["fsliced"]),
    ]
    # Pad so that the sliced piece lands at (dst_r0, dst_c0) and the canvas
    # becomes (H, W). End-pads finish the canvas to the right and bottom.
    pads = np.asarray([
        0, 0, dst_r0, dst_c0,
        0, 0, H - dst_r0 - rh, W - dst_c0 - rw
    ], dtype=np.int64)
    nodes += [
        _const("fpp", pads),
        oh.make_node("Pad", ["fsliced", "fpp"], ["output"], mode="constant"),
    ]
    return _make_model(nodes, "framed_translate_net")


# --- Upscale via Resize (nearest), 0 MACs ----------------------------------

def make_upscale(f: int, in_h: int, in_w: int):
    """Nearest-neighbor upscale of the content region [0:in_h, 0:in_w] by
    factor f. The real content region is first sliced out, then Resize
    upsamples each axis by f, then Pad brings it back to (H, W)."""
    nodes = _crop_to_content("input", "content", in_h, in_w, "0")
    # Resize: needs 'roi' and 'scales' (empty roi). Use sizes for deterministic output.
    nodes += [
        _const("rsz_roi", np.asarray([], dtype=np.float32)),
        _const("rsz_scales", np.asarray([], dtype=np.float32)),
        _const("rsz_sizes", np.asarray([1, C, in_h * f, in_w * f], dtype=np.int64)),
        oh.make_node(
            "Resize",
            ["content", "rsz_roi", "rsz_scales", "rsz_sizes"],
            ["upscaled"],
            mode="nearest",
            coordinate_transformation_mode="asymmetric",
            nearest_mode="floor",
        ),
    ]
    nodes += _pad_to_canvas("upscaled", "output", in_h * f, in_w * f, "p")
    return _make_model(nodes, "upscale_net")


# --- Downscale via Slice with stride, 0 MACs -------------------------------

def make_downscale(f: int, in_h: int, in_w: int):
    """Take every f-th pixel along H and W from the [0:in_h, 0:in_w] region."""
    out_h = in_h // f
    out_w = in_w // f
    nodes = [
        _const("ds_s", np.asarray([0, 0], dtype=np.int64)),
        _const("ds_e", np.asarray([in_h, in_w], dtype=np.int64)),
        _const("ds_a", np.asarray([2, 3], dtype=np.int64)),
        _const("ds_p", np.asarray([f, f], dtype=np.int64)),
        oh.make_node("Slice", ["input", "ds_s", "ds_e", "ds_a", "ds_p"], ["downsampled"]),
    ]
    nodes += _pad_to_canvas("downsampled", "output", out_h, out_w, "p")
    return _make_model(nodes, "downscale_net")


# --- Tile via Tile op, 0 MACs ----------------------------------------------

def make_tile(tr: int, tc: int, in_h: int, in_w: int):
    """Slice the content region then Tile it (tr, tc) times along H, W."""
    nodes = _crop_to_content("input", "content", in_h, in_w, "0")
    repeats = np.asarray([1, 1, tr, tc], dtype=np.int64)
    nodes += [
        _const("tile_r", repeats),
        oh.make_node("Tile", ["content", "tile_r"], ["tiled"]),
    ]
    nodes += _pad_to_canvas("tiled", "output", in_h * tr, in_w * tc, "p")
    return _make_model(nodes, "tile_net")


# --- Concatenate-with-mirror: output = [input | flip(input)] ---------------

def make_concat_flip_w(in_h: int, in_w: int):
    """Output (h, 2w) = [input (h, w) | flip_w(input)]. 0 MACs."""
    nodes = _crop_to_content("input", "content", in_h, in_w, "0")
    nodes += _slice_reverse_nodes([3], [in_w], "content", "flipped", "fw")
    nodes += [oh.make_node("Concat", ["content", "flipped"], ["concated"], axis=3)]
    nodes += _pad_to_canvas("concated", "output", in_h, in_w * 2, "p")
    return _make_model(nodes, "concat_flip_w")


def make_concat_flip_h(in_h: int, in_w: int):
    """Output (2h, w) = [input; flip_h(input)]. 0 MACs."""
    nodes = _crop_to_content("input", "content", in_h, in_w, "0")
    nodes += _slice_reverse_nodes([2], [in_h], "content", "flipped", "fh")
    nodes += [oh.make_node("Concat", ["content", "flipped"], ["concated"], axis=2)]
    nodes += _pad_to_canvas("concated", "output", in_h * 2, in_w, "p")
    return _make_model(nodes, "concat_flip_h")


def make_concat_rot180_w(in_h: int, in_w: int):
    """Output (h, 2w) = [input | rot180(input)]. 0 MACs."""
    nodes = _crop_to_content("input", "content", in_h, in_w, "0")
    nodes += _slice_reverse_nodes([2, 3], [in_h, in_w], "content", "rotated", "r")
    nodes += [oh.make_node("Concat", ["content", "rotated"], ["concated"], axis=3)]
    nodes += _pad_to_canvas("concated", "output", in_h, in_w * 2, "p")
    return _make_model(nodes, "concat_rot180_w")


# --- Quadrant mirror: 2h × 2w output from h × w input ----------------------

def make_quadrant_mirror(in_h: int, in_w: int):
    """Top-left    = input
       Top-right   = flip_w(input)
       Bottom-left = flip_h(input)
       Bottom-right= rot180(input)  ( = flip_h(flip_w(input)) )
    Built in two Concat stages. 0 MACs."""
    nodes = _crop_to_content("input", "content", in_h, in_w, "0")
    # Build top row: [content | flip_w(content)]
    nodes += _slice_reverse_nodes([3], [in_w], "content", "fw", "fw")
    nodes += [oh.make_node("Concat", ["content", "fw"], ["top_row"], axis=3)]
    # Build bottom row: [flip_h(content) | rot180(content)]
    nodes += _slice_reverse_nodes([2], [in_h], "content", "fh", "fh")
    nodes += _slice_reverse_nodes([2, 3], [in_h, in_w], "content", "fr", "fr")
    nodes += [oh.make_node("Concat", ["fh", "fr"], ["bot_row"], axis=3)]
    # Stack rows along H
    nodes += [oh.make_node("Concat", ["top_row", "bot_row"], ["full"], axis=2)]
    nodes += _pad_to_canvas("full", "output", in_h * 2, in_w * 2, "p")
    return _make_model(nodes, "quadrant_mirror")


# --- Row/column broadcast: every row = row 0 (or col 0) of input -----------

def make_row_broadcast(in_h: int, in_w: int, row_idx: int = 0):
    """Output[r, c] = input[row_idx, c] for r in [0..in_h), c in [0..in_w)."""
    # Slice a single row, then Tile along H axis to replicate.
    starts = np.asarray([row_idx], dtype=np.int64)
    ends = np.asarray([row_idx + 1], dtype=np.int64)
    axes = np.asarray([2], dtype=np.int64)
    nodes = [
        _const("rb_s", starts),
        _const("rb_e", ends),
        _const("rb_a", axes),
        oh.make_node("Slice", ["input", "rb_s", "rb_e", "rb_a"], ["one_row"]),
    ]
    # After slice: shape (1, C, 1, W). Tile along H to get (1, C, in_h, W).
    nodes += [
        _const("rb_r", np.asarray([1, 1, in_h, 1], dtype=np.int64)),
        oh.make_node("Tile", ["one_row", "rb_r"], ["tiled"]),
    ]
    # Now crop width to in_w, then pad.
    nodes += [
        _const("rb_cs", np.asarray([0, 0], dtype=np.int64)),
        _const("rb_ce", np.asarray([in_h, in_w], dtype=np.int64)),
        _const("rb_ca", np.asarray([2, 3], dtype=np.int64)),
        oh.make_node("Slice", ["tiled", "rb_cs", "rb_ce", "rb_ca"], ["cropped"]),
    ]
    nodes += _pad_to_canvas("cropped", "output", in_h, in_w, "p")
    return _make_model(nodes, "row_broadcast")


def make_col_broadcast(in_h: int, in_w: int, col_idx: int = 0):
    """Output[r, c] = input[r, col_idx]."""
    starts = np.asarray([col_idx], dtype=np.int64)
    ends = np.asarray([col_idx + 1], dtype=np.int64)
    axes = np.asarray([3], dtype=np.int64)
    nodes = [
        _const("cb_s", starts),
        _const("cb_e", ends),
        _const("cb_a", axes),
        oh.make_node("Slice", ["input", "cb_s", "cb_e", "cb_a"], ["one_col"]),
    ]
    nodes += [
        _const("cb_r", np.asarray([1, 1, 1, in_w], dtype=np.int64)),
        oh.make_node("Tile", ["one_col", "cb_r"], ["tiled"]),
    ]
    nodes += [
        _const("cb_cs", np.asarray([0, 0], dtype=np.int64)),
        _const("cb_ce", np.asarray([in_h, in_w], dtype=np.int64)),
        _const("cb_ca", np.asarray([2, 3], dtype=np.int64)),
        oh.make_node("Slice", ["tiled", "cb_cs", "cb_ce", "cb_ca"], ["cropped"]),
    ]
    nodes += _pad_to_canvas("cropped", "output", in_h, in_w, "p")
    return _make_model(nodes, "col_broadcast")


# --- Crop + transform composition -----------------------------------------

def _crop_nodes(r0: int, c0: int, out_h: int, out_w: int,
                in_name: str, out_name: str, suf: str = "ct"):
    return [
        _const(f"{suf}_s", np.asarray([r0, c0], dtype=np.int64)),
        _const(f"{suf}_e", np.asarray([r0 + out_h, c0 + out_w], dtype=np.int64)),
        _const(f"{suf}_a", np.asarray([2, 3], dtype=np.int64)),
        oh.make_node("Slice", [in_name, f"{suf}_s", f"{suf}_e", f"{suf}_a"], [out_name]),
    ]


def make_crop_then_color(r0, c0, out_h, out_w, color_map):
    """Crop and apply color remap (Gather on channels)."""
    # Build crop nodes into 'cropped', then gather channels, then pad.
    nodes = _crop_nodes(r0, c0, out_h, out_w, "input", "cropped", "ctc")

    inv: Dict[int, int] = {}
    for s, d in color_map.items():
        inv[d] = s
    idx = np.arange(C, dtype=np.int64)
    mask = np.ones(C, dtype=np.float32)
    src_remapped_away = {s for s, d in color_map.items() if s != d}
    for d in range(C):
        if d in inv:
            idx[d] = inv[d]
        elif d in src_remapped_away:
            idx[d] = 0
            mask[d] = 0.0
        else:
            idx[d] = d
    gi = _const("gi_ctc", idx)
    if mask.min() == 1.0:
        gnode = oh.make_node("Gather", ["cropped", "gi_ctc"], ["colored"], axis=1)
        nodes += [gi, gnode]
    else:
        gnode = oh.make_node("Gather", ["cropped", "gi_ctc"], ["g_ctc"], axis=1)
        mk = _const("cm_ctc", mask.reshape(1, C, 1, 1))
        mul = oh.make_node("Mul", ["g_ctc", "cm_ctc"], ["colored"])
        nodes += [gi, gnode, mk, mul]

    nodes += _pad_to_canvas("colored", "output", out_h, out_w, "pctc")
    return _make_model(nodes, "crop_color")


def make_pad_with_border(grid_h, grid_w, border: int, fill_color: int):
    """Add a border of `border` thickness around the grid, filled with fill_color.
    Output shape = (grid_h + 2*border, grid_w + 2*border).

    Approach: write a full constant of shape (1, C, grid_h+2b, grid_w+2b) where
    fill-color channel is all ones (for the border region) and zero inside;
    then slice the content from input and put it inside. This uses 2 Pad's
    with different constant values... more cleanly: we build the output by
    concatenating a pre-filled border around the content.

    Cheaper: do Pad with mode='constant' and value=(1-hot tensor via channel).
    Since channel axis is not padded, Pad adds zeros everywhere. We need the
    border cells to have channel `fill_color` = 1. So we mask them on via Add
    with a precomputed border mask (only border cells non-zero, only on the
    fill_color channel).
    """
    oh_ = grid_h + 2 * border
    ow_ = grid_w + 2 * border
    if oh_ > H or ow_ > W:
        return None

    # Slice content, pad with zeros to (oh_, ow_), then add a border mask that
    # has 1's at fill_color channel in the border region.
    nodes = _crop_to_content("input", "content", grid_h, grid_w, "pb0")
    # Pad content with border zeros:
    pads_border = np.asarray([
        0, 0, border, border,
        0, 0, border, border
    ], dtype=np.int64)
    nodes += [
        _const("pb_pads", pads_border),
        oh.make_node("Pad", ["content", "pb_pads"], ["padded"], mode="constant"),
    ]
    # Now the result has shape (1, C, oh_, ow_) with zeros where the border goes.
    # Build a mask (1, C, oh_, ow_) with 1 at [fill_color, border-cells]:
    mask = np.zeros((1, C, oh_, ow_), dtype=np.float32)
    if 0 <= fill_color < C:
        # border rows and cols
        mask[0, fill_color, :border, :] = 1.0
        mask[0, fill_color, -border:, :] = 1.0
        mask[0, fill_color, :, :border] = 1.0
        mask[0, fill_color, :, -border:] = 1.0
    nodes += [
        _const("pb_mask", mask),
        oh.make_node("Add", ["padded", "pb_mask"], ["bordered"]),
    ]
    # Pad to full canvas (H, W)
    nodes += _pad_to_canvas("bordered", "output", oh_, ow_, "pbfinal")
    return _make_model(nodes, "pad_with_border")


# --- Inset: place input inside a larger canvas at fixed position, rest=0 ---

def make_inset(grid_h, grid_w, offset_r, offset_w):
    """Move input's content [0:grid_h, 0:grid_w] to [offset_r:+grid_h, offset_w:+grid_w]
    in the output. Everything else = 0. Essentially a framed translation."""
    return make_framed_translation(grid_h + offset_r, grid_w + offset_w,
                                    offset_r, offset_w)


# --- Composition: transform then color remap --------------------------------

def make_transform_plus_color(base_fn, color_map: Dict[int, int]):
    """Run a transform model, then Gather on channel axis for color remap.
    We splice the two subgraphs together by renaming the base's 'output' to
    'mid' and feeding 'mid' into a Gather.
    """
    base = base_fn()
    base_nodes = [copy.deepcopy(n) for n in base.graph.node]
    for node in base_nodes:
        for i, o in enumerate(node.output):
            if o == "output":
                node.output[i] = "mid"

    inv: Dict[int, int] = {}
    for s, d in color_map.items():
        inv[d] = s
    idx = np.arange(C, dtype=np.int64)
    mask = np.ones(C, dtype=np.float32)
    src_remapped_away = {s for s, d in color_map.items() if s != d}
    for d in range(C):
        if d in inv:
            idx[d] = inv[d]
        elif d in src_remapped_away:
            idx[d] = 0
            mask[d] = 0.0
        else:
            idx[d] = d
    gi = _const("gi2", idx)
    if mask.min() == 1.0:
        gnode = oh.make_node("Gather", ["mid", "gi2"], ["output"], axis=1)
        return _make_model(base_nodes + [gi, gnode], "xform_color")
    gnode = oh.make_node("Gather", ["mid", "gi2"], ["gathered2"], axis=1)
    mk = _const("cm2", mask.reshape(1, C, 1, 1))
    mul = oh.make_node("Mul", ["gathered2", "cm2"], ["output"])
    return _make_model(base_nodes + [gi, gnode, mk, mul], "xform_color_masked")


# =============================================================================
# Detectors
# =============================================================================

def all_same_io_shape(pairs) -> bool:
    return all(np.asarray(p["input"]).shape == np.asarray(p["output"]).shape
               for p in pairs)

def all_square_io(pairs) -> bool:
    return all(np.asarray(p["input"]).shape[0] == np.asarray(p["input"]).shape[1]
               for p in pairs)

def fixed_input_size(pairs):
    if not pairs:
        return None
    first = np.asarray(pairs[0]["input"]).shape
    for p in pairs[1:]:
        if np.asarray(p["input"]).shape != first:
            return None
    return first

def detect_identity(pairs) -> bool:
    for p in pairs:
        ig = np.asarray(p["input"]); og = np.asarray(p["output"])
        if ig.shape != og.shape or not np.array_equal(ig, og):
            return False
    return True

def detect_const(pairs):
    if not pairs:
        return None
    first = np.asarray(pairs[0]["output"])
    for p in pairs[1:]:
        o = np.asarray(p["output"])
        if o.shape != first.shape or not np.array_equal(o, first):
            return None
    ins = {str(p["input"]) for p in pairs}
    if len(ins) < 2 and detect_identity(pairs):
        return None
    return first

def detect_rotation(pairs):
    for label, k in [("180", 2), ("90cw", -1), ("90ccw", 1)]:
        ok = True
        for p in pairs:
            ig = np.asarray(p["input"]); og = np.asarray(p["output"])
            r = np.rot90(ig, k)
            if r.shape != og.shape or not np.array_equal(r, og):
                ok = False; break
        if ok:
            return label
    return None

def detect_flip(pairs):
    for name, fn in [("h", np.flipud), ("w", np.fliplr),
                     ("both", lambda a: np.flipud(np.fliplr(a)))]:
        ok = True
        for p in pairs:
            ig = np.asarray(p["input"]); og = np.asarray(p["output"])
            f = fn(ig)
            if f.shape != og.shape or not np.array_equal(f, og):
                ok = False; break
        if ok:
            return name
    return None

def detect_transpose(pairs):
    for p in pairs:
        ig = np.asarray(p["input"]); og = np.asarray(p["output"])
        if ig.T.shape != og.shape or not np.array_equal(ig.T, og):
            return False
    return True

def detect_anti_transpose(pairs):
    for p in pairs:
        ig = np.asarray(p["input"]); og = np.asarray(p["output"])
        exp = np.flipud(np.fliplr(ig.T))
        if exp.shape != og.shape or not np.array_equal(exp, og):
            return False
    return True

def detect_color_map(pairs):
    if not all_same_io_shape(pairs):
        return None
    mapping: Dict[int, int] = {}
    for p in pairs:
        ig = np.asarray(p["input"]).flatten().tolist()
        og = np.asarray(p["output"]).flatten().tolist()
        for a, b in zip(ig, og):
            a, b = int(a), int(b)
            if a in mapping and mapping[a] != b:
                return None
            mapping[a] = b
    if all(k == v for k, v in mapping.items()):
        return None
    return mapping

def detect_rotation_plus_color(pairs):
    for rot_label, k in [("180", 2), ("90cw", -1), ("90ccw", 1)]:
        mapping: Dict[int, int] = {}
        ok = True
        for p in pairs:
            ig = np.asarray(p["input"]); og = np.asarray(p["output"])
            r = np.rot90(ig, k)
            if r.shape != og.shape:
                ok = False; break
            for a, b in zip(r.flatten().tolist(), og.flatten().tolist()):
                a, b = int(a), int(b)
                if a in mapping and mapping[a] != b:
                    ok = False; break
                mapping[a] = b
            if not ok: break
        if ok and any(k2 != v for k2, v in mapping.items()):
            return rot_label, mapping
    return None

def detect_flip_plus_color(pairs):
    for name, fn in [("h", np.flipud), ("w", np.fliplr),
                     ("both", lambda a: np.flipud(np.fliplr(a)))]:
        mapping: Dict[int, int] = {}
        ok = True
        for p in pairs:
            ig = np.asarray(p["input"]); og = np.asarray(p["output"])
            f = fn(ig)
            if f.shape != og.shape:
                ok = False; break
            for a, b in zip(f.flatten().tolist(), og.flatten().tolist()):
                a, b = int(a), int(b)
                if a in mapping and mapping[a] != b:
                    ok = False; break
                mapping[a] = b
            if not ok: break
        if ok and any(k2 != v for k2, v in mapping.items()):
            return name, mapping
    return None

def detect_transpose_plus_color(pairs):
    mapping: Dict[int, int] = {}
    for p in pairs:
        ig = np.asarray(p["input"]); og = np.asarray(p["output"])
        if ig.T.shape != og.shape:
            return None
        for a, b in zip(ig.T.flatten().tolist(), og.flatten().tolist()):
            a, b = int(a), int(b)
            if a in mapping and mapping[a] != b:
                return None
            mapping[a] = b
    if any(k != v for k, v in mapping.items()):
        return mapping
    return None

def detect_fixed_crop(pairs):
    if not pairs:
        return None
    first_in = np.asarray(pairs[0]["input"])
    first_out = np.asarray(pairs[0]["output"])
    oh_, ow_ = first_out.shape
    ih, iw = first_in.shape
    if oh_ > ih or ow_ > iw:
        return None
    for r0 in range(ih - oh_ + 1):
        for c0 in range(iw - ow_ + 1):
            if not np.array_equal(first_in[r0:r0+oh_, c0:c0+ow_], first_out):
                continue
            ok = True
            for p in pairs[1:]:
                ig = np.asarray(p["input"]); og = np.asarray(p["output"])
                if og.shape != (oh_, ow_):
                    ok = False; break
                if r0 + oh_ > ig.shape[0] or c0 + ow_ > ig.shape[1]:
                    ok = False; break
                if not np.array_equal(ig[r0:r0+oh_, c0:c0+ow_], og):
                    ok = False; break
            if ok:
                return (r0, c0, oh_, ow_)
    return None

def detect_translation(pairs):
    if not all_same_io_shape(pairs):
        return None
    first_in = np.asarray(pairs[0]["input"])
    ih, iw = first_in.shape
    # Limit search to reasonable shifts to keep this fast (|dr|, |dc| ≤ 10).
    rng_r = min(ih, 11)
    rng_c = min(iw, 11)
    for dr in range(-rng_r + 1, rng_r):
        for dc in range(-rng_c + 1, rng_c):
            if dr == 0 and dc == 0:
                continue
            ok = True
            for p in pairs:
                ig = np.asarray(p["input"]); og = np.asarray(p["output"])
                if ig.shape != og.shape:
                    ok = False; break
                hh, ww = ig.shape
                shifted = np.zeros_like(ig)
                src_r0 = max(0, -dr); src_r1 = min(hh, hh - dr)
                src_c0 = max(0, -dc); src_c1 = min(ww, ww - dc)
                dst_r0 = max(0, dr);  dst_c0 = max(0, dc)
                rh = src_r1 - src_r0; rw = src_c1 - src_c0
                if rh > 0 and rw > 0:
                    shifted[dst_r0:dst_r0+rh, dst_c0:dst_c0+rw] = \
                        ig[src_r0:src_r1, src_c0:src_c1]
                if not np.array_equal(shifted, og):
                    ok = False; break
            if ok:
                return (dr, dc)
    return None

def detect_color_filter(pairs):
    """Set of colours that get replaced with 0, everything else unchanged."""
    if not all_same_io_shape(pairs):
        return None
    candidate: Optional[set] = None
    for p in pairs:
        ig = np.asarray(p["input"]); og = np.asarray(p["output"])
        diff = (ig != og)
        if not np.any(diff):
            continue
        if not np.all(og[diff] == 0):
            return None
        local = set(int(x) for x in ig[diff].tolist())
        candidate = local if candidate is None else candidate | local
    if not candidate:
        return None
    for p in pairs:
        ig = np.asarray(p["input"]).copy()
        for c_ in candidate:
            ig[ig == c_] = 0
        if not np.array_equal(ig, np.asarray(p["output"])):
            return None
    return candidate


def detect_upscale(pairs):
    """Nearest-neighbor integer upscale by factor f; same f for all pairs."""
    if not pairs:
        return None
    for f in [2, 3, 4, 5]:
        ok = True
        for p in pairs:
            ig = np.asarray(p["input"]); og = np.asarray(p["output"])
            ih, iw = ig.shape
            oh_, ow_ = og.shape
            if oh_ != ih * f or ow_ != iw * f:
                ok = False; break
            if ih * f > H or iw * f > W:
                ok = False; break
            scaled = np.repeat(np.repeat(ig, f, axis=0), f, axis=1)
            if not np.array_equal(scaled, og):
                ok = False; break
        if ok:
            return f
    return None


def detect_downscale(pairs):
    """Downsample by taking every f-th pixel; same f for all pairs."""
    if not pairs:
        return None
    for f in [2, 3, 4, 5]:
        ok = True
        for p in pairs:
            ig = np.asarray(p["input"]); og = np.asarray(p["output"])
            ih, iw = ig.shape
            oh_, ow_ = og.shape
            if ih != oh_ * f or iw != ow_ * f:
                ok = False; break
            if not np.array_equal(ig[::f, ::f], og):
                ok = False; break
        if ok:
            return f
    return None


def detect_tile(pairs):
    """Output is input tiled tr × tc times."""
    if not pairs:
        return None
    # Deduce (tr, tc) from first pair.
    first_in = np.asarray(pairs[0]["input"])
    first_out = np.asarray(pairs[0]["output"])
    ih, iw = first_in.shape
    oh_, ow_ = first_out.shape
    if ih == 0 or iw == 0:
        return None
    if oh_ % ih != 0 or ow_ % iw != 0:
        return None
    tr, tc = oh_ // ih, ow_ // iw
    if tr * ih > H or tc * iw > W:
        return None
    if tr == 1 and tc == 1:
        return None
    for p in pairs:
        ig = np.asarray(p["input"]); og = np.asarray(p["output"])
        ih2, iw2 = ig.shape
        if og.shape != (ih2 * tr, iw2 * tc):
            return None
        tiled = np.tile(ig, (tr, tc))
        if not np.array_equal(tiled, og):
            return None
    return (tr, tc)


def detect_concat_flip_w(pairs):
    """Output = [input | fliplr(input)]."""
    if not pairs: return None
    for p in pairs:
        ig = np.asarray(p["input"]); og = np.asarray(p["output"])
        ih, iw = ig.shape
        if og.shape != (ih, iw * 2): return None
        if not np.array_equal(np.concatenate([ig, np.fliplr(ig)], axis=1), og):
            return None
    return True


def detect_concat_flip_h(pairs):
    if not pairs: return None
    for p in pairs:
        ig = np.asarray(p["input"]); og = np.asarray(p["output"])
        ih, iw = ig.shape
        if og.shape != (ih * 2, iw): return None
        if not np.array_equal(np.concatenate([ig, np.flipud(ig)], axis=0), og):
            return None
    return True


def detect_concat_rot180_w(pairs):
    if not pairs: return None
    for p in pairs:
        ig = np.asarray(p["input"]); og = np.asarray(p["output"])
        ih, iw = ig.shape
        if og.shape != (ih, iw * 2): return None
        if not np.array_equal(np.concatenate([ig, np.rot90(ig, 2)], axis=1), og):
            return None
    return True


def detect_quadrant_mirror(pairs):
    """Output (2h, 2w) = quadrant mirror of input (h, w)."""
    if not pairs: return None
    for p in pairs:
        ig = np.asarray(p["input"]); og = np.asarray(p["output"])
        ih, iw = ig.shape
        if og.shape != (ih * 2, iw * 2): return None
        top = np.concatenate([ig, np.fliplr(ig)], axis=1)
        bot = np.concatenate([np.flipud(ig), np.rot90(ig, 2)], axis=1)
        expected = np.concatenate([top, bot], axis=0)
        if not np.array_equal(expected, og): return None
    return True


def detect_row_broadcast(pairs):
    """Every row of output == some specific row of input (same index for all pairs)."""
    if not all_same_io_shape(pairs):
        return None
    chosen_row = None
    for p in pairs:
        ig = np.asarray(p["input"]); og = np.asarray(p["output"])
        ih, iw = ig.shape
        matched = None
        for r in range(ih):
            if np.all(og == ig[r, :]):
                matched = r; break
        if matched is None:
            return None
        if chosen_row is None:
            chosen_row = matched
        elif chosen_row != matched:
            return None
    return chosen_row


def detect_col_broadcast(pairs):
    if not all_same_io_shape(pairs):
        return None
    chosen_col = None
    for p in pairs:
        ig = np.asarray(p["input"]); og = np.asarray(p["output"])
        ih, iw = ig.shape
        matched = None
        for c in range(iw):
            if np.all(og == ig[:, c:c+1]):
                matched = c; break
        if matched is None:
            return None
        if chosen_col is None:
            chosen_col = matched
        elif chosen_col != matched:
            return None
    return chosen_col


def detect_crop_plus_color(pairs):
    """Output = crop with same offset + a color remap applied."""
    if not pairs: return None
    first_in = np.asarray(pairs[0]["input"])
    first_out = np.asarray(pairs[0]["output"])
    oh_, ow_ = first_out.shape
    ih, iw = first_in.shape
    if oh_ > ih or ow_ > iw:
        return None
    for r0 in range(ih - oh_ + 1):
        for c0 in range(iw - ow_ + 1):
            # Aggregate cmap across ALL pairs so partial colors don't bite us.
            cmap: Dict[int, int] = {}
            valid = True
            for p in pairs:
                ig = np.asarray(p["input"]); og = np.asarray(p["output"])
                if og.shape != (oh_, ow_):
                    valid = False; break
                if r0 + oh_ > ig.shape[0] or c0 + ow_ > ig.shape[1]:
                    valid = False; break
                region = ig[r0:r0+oh_, c0:c0+ow_]
                for a, b in zip(region.flatten().tolist(), og.flatten().tolist()):
                    a, b = int(a), int(b)
                    if a in cmap and cmap[a] != b:
                        valid = False; break
                    cmap[a] = b
                if not valid: break
            if not valid:
                continue
            if all(k == v for k, v in cmap.items()):
                continue  # pure crop, let detect_fixed_crop handle it
            # Final verification (paranoia)
            ok = True
            for p in pairs:
                ig = np.asarray(p["input"]); og = np.asarray(p["output"])
                region2 = ig[r0:r0+oh_, c0:c0+ow_]
                mapped = region2.copy()
                for k, v in cmap.items():
                    mapped[region2 == k] = v
                if not np.array_equal(mapped, og):
                    ok = False; break
            if ok:
                return (r0, c0, oh_, ow_, cmap)
    return None


def detect_border_add(pairs):
    """Output = input with a uniform border of thickness b, color fill."""
    if not pairs: return None
    first_in = np.asarray(pairs[0]["input"])
    first_out = np.asarray(pairs[0]["output"])
    ih, iw = first_in.shape
    oh_, ow_ = first_out.shape
    if (oh_ - ih) != (ow_ - iw):
        return None
    delta = oh_ - ih
    if delta <= 0 or delta % 2 != 0:
        return None
    b = delta // 2
    if not np.array_equal(first_out[b:b+ih, b:b+iw], first_in):
        return None
    vals = set(first_out[:b, :].flatten().tolist())
    vals |= set(first_out[-b:, :].flatten().tolist())
    vals |= set(first_out[:, :b].flatten().tolist())
    vals |= set(first_out[:, -b:].flatten().tolist())
    if len(vals) != 1:
        return None
    fill = int(vals.pop())
    for p in pairs[1:]:
        ig = np.asarray(p["input"]); og = np.asarray(p["output"])
        ih2, iw2 = ig.shape
        if og.shape != (ih2 + 2*b, iw2 + 2*b):
            return None
        if not np.array_equal(og[b:b+ih2, b:b+iw2], ig):
            return None
        m = np.zeros(og.shape, dtype=bool)
        m[:b, :] = True; m[-b:, :] = True
        m[:, :b] = True; m[:, -b:] = True
        if not np.all(og[m] == fill):
            return None
    return (b, fill)


# =============================================================================
# Orchestrator: try all applicable candidates, pick cheapest correct one
# =============================================================================

def _candidates(train_pairs, all_pairs):
    """`train_pairs` drives pattern detection (the transformation rule must be
    visible there). `all_pairs` — which includes test and arc-gen — is used
    only to decide whether input sizes are truly fixed across all examples,
    so framed builders don't over-commit to a size that test/arc-gen break."""
    fixed_size = fixed_input_size(all_pairs)
    is_full_canvas = (fixed_size == (30, 30))
    same_shape = all_same_io_shape(train_pairs)

    # 1. Identity
    if detect_identity(train_pairs):
        yield "identity", make_identity

    # 2. Rotations / flips / transposes
    rot = detect_rotation(train_pairs)
    if rot == "180":
        if is_full_canvas:
            yield "rot180", make_rot_180
        if fixed_size is not None:
            h, w = fixed_size
            yield f"rot180_framed", (lambda h=h, w=w: make_framed_rot180(h, w))
    if rot == "90cw":
        if is_full_canvas:
            yield "rot90cw", make_rot_90_cw
        if fixed_size is not None:
            h, w = fixed_size
            yield f"rot90cw_framed", (lambda h=h, w=w: make_framed_rot_90_cw(h, w))
    if rot == "90ccw":
        if is_full_canvas:
            yield "rot90ccw", make_rot_90_ccw
        if fixed_size is not None:
            h, w = fixed_size
            yield f"rot90ccw_framed", (lambda h=h, w=w: make_framed_rot_90_ccw(h, w))

    flip = detect_flip(train_pairs)
    if flip == "h":
        if is_full_canvas:
            yield "flip_h", make_flip_h
        if fixed_size is not None:
            h, w = fixed_size
            yield f"flip_h_framed", (lambda h=h, w=w: make_framed_flip_h(h, w))
    if flip == "w":
        if is_full_canvas:
            yield "flip_w", make_flip_w
        if fixed_size is not None:
            h, w = fixed_size
            yield f"flip_w_framed", (lambda h=h, w=w: make_framed_flip_w(h, w))
    if flip == "both":
        if is_full_canvas:
            yield "flip_both", make_flip_both
        if fixed_size is not None:
            h, w = fixed_size
            yield f"flip_both_framed", (lambda h=h, w=w: make_framed_rot180(h, w))

    if detect_transpose(train_pairs):
        if is_full_canvas:
            yield "transpose", make_transpose_hw
        if fixed_size is not None:
            h, w = fixed_size
            yield f"transpose_framed", (lambda h=h, w=w: make_framed_transpose(h, w))

    if detect_anti_transpose(train_pairs) and is_full_canvas:
        yield "anti_transpose", make_anti_transpose

    # 3. Color map
    cmap = detect_color_map(train_pairs)
    if cmap is not None and same_shape:
        yield "color_remap", (lambda cm=cmap: make_color_remap_gather(cm))

    # 4. Compositions: rotation/flip/transpose + color
    rot_col = detect_rotation_plus_color(train_pairs)
    if rot_col is not None and same_shape:
        rl, cm = rot_col
        if rl == "180":
            if is_full_canvas:
                yield "rot180_color", (lambda cm=cm: make_transform_plus_color(make_rot_180, cm))
            if fixed_size is not None:
                h, w = fixed_size
                yield "rot180_framed_color", (
                    lambda h=h, w=w, cm=cm: make_transform_plus_color(
                        lambda h=h, w=w: make_framed_rot180(h, w), cm
                    )
                )
        elif rl == "90cw":
            if is_full_canvas:
                yield "rot90cw_color", (lambda cm=cm: make_transform_plus_color(make_rot_90_cw, cm))
            if fixed_size is not None:
                h, w = fixed_size
                yield "rot90cw_framed_color", (
                    lambda h=h, w=w, cm=cm: make_transform_plus_color(
                        lambda h=h, w=w: make_framed_rot_90_cw(h, w), cm
                    )
                )
        elif rl == "90ccw":
            if is_full_canvas:
                yield "rot90ccw_color", (lambda cm=cm: make_transform_plus_color(make_rot_90_ccw, cm))
            if fixed_size is not None:
                h, w = fixed_size
                yield "rot90ccw_framed_color", (
                    lambda h=h, w=w, cm=cm: make_transform_plus_color(
                        lambda h=h, w=w: make_framed_rot_90_ccw(h, w), cm
                    )
                )

    flip_col = detect_flip_plus_color(train_pairs)
    if flip_col is not None and same_shape:
        fl, cm = flip_col
        if is_full_canvas:
            base_full = {"h": make_flip_h, "w": make_flip_w, "both": make_flip_both}[fl]
            yield f"flip_{fl}_color", (lambda b=base_full, cm=cm: make_transform_plus_color(b, cm))
        if fixed_size is not None:
            h, w = fixed_size
            base_framed = {
                "h":    lambda h=h, w=w: make_framed_flip_h(h, w),
                "w":    lambda h=h, w=w: make_framed_flip_w(h, w),
                "both": lambda h=h, w=w: make_framed_rot180(h, w),
            }[fl]
            yield f"flip_{fl}_framed_color", (
                lambda base_framed=base_framed, cm=cm:
                    make_transform_plus_color(base_framed, cm)
            )

    trp_col = detect_transpose_plus_color(train_pairs)
    if trp_col is not None:
        if is_full_canvas:
            yield "transpose_color", (lambda cm=trp_col: make_transform_plus_color(make_transpose_hw, cm))
        if fixed_size is not None:
            h, w = fixed_size
            yield "transpose_framed_color", (
                lambda h=h, w=w, cm=trp_col:
                    make_transform_plus_color(lambda h=h, w=w: make_framed_transpose(h, w), cm)
            )

    # 5. Color filter
    cf = detect_color_filter(train_pairs)
    if cf is not None and same_shape:
        mapping = {c_: 0 for c_ in cf}
        yield "color_filter", (lambda cm=mapping: make_color_remap_gather(cm))

    # 6. Fixed crop
    crop = detect_fixed_crop(train_pairs)
    if crop is not None:
        r0, c0, oh_, ow_ = crop
        yield f"crop_{r0}_{c0}", (
            lambda r0=r0, c0=c0, oh_=oh_, ow_=ow_: make_crop(r0, c0, oh_, ow_)
        )

    # 7. Fixed translation
    tr = detect_translation(train_pairs)
    if tr is not None:
        dr, dc = tr
        if is_full_canvas:
            yield f"translate_{dr}_{dc}", (lambda dr=dr, dc=dc: make_translation(dr, dc))
        if fixed_size is not None:
            h, w = fixed_size
            yield f"translate_framed_{dr}_{dc}", (
                lambda h=h, w=w, dr=dr, dc=dc: make_framed_translation(h, w, dr, dc)
            )

    # 8. Upscale (nearest-neighbor, integer factor)
    if fixed_size is not None:
        up_f = detect_upscale(train_pairs)
        if up_f is not None:
            h, w = fixed_size
            yield f"upscale_{up_f}x", (
                lambda f=up_f, h=h, w=w: make_upscale(f, h, w)
            )

    # 9. Downscale
    if fixed_size is not None:
        down_f = detect_downscale(train_pairs)
        if down_f is not None:
            h, w = fixed_size
            yield f"downscale_{down_f}x", (
                lambda f=down_f, h=h, w=w: make_downscale(f, h, w)
            )

    # 10. Tile
    if fixed_size is not None:
        tile = detect_tile(train_pairs)
        if tile is not None:
            tr_, tc_ = tile
            h, w = fixed_size
            yield f"tile_{tr_}x{tc_}", (
                lambda tr_=tr_, tc_=tc_, h=h, w=w: make_tile(tr_, tc_, h, w)
            )

    # 10b. Concat-mirror patterns (fixed size only)
    if fixed_size is not None:
        h, w = fixed_size
        if detect_concat_flip_w(train_pairs) and h * 1 <= H and w * 2 <= W:
            yield "concat_flip_w", (lambda h=h, w=w: make_concat_flip_w(h, w))
        if detect_concat_flip_h(train_pairs) and h * 2 <= H and w <= W:
            yield "concat_flip_h", (lambda h=h, w=w: make_concat_flip_h(h, w))
        if detect_concat_rot180_w(train_pairs) and h <= H and w * 2 <= W:
            yield "concat_rot180_w", (lambda h=h, w=w: make_concat_rot180_w(h, w))
        if detect_quadrant_mirror(train_pairs) and h * 2 <= H and w * 2 <= W:
            yield "quadrant_mirror", (lambda h=h, w=w: make_quadrant_mirror(h, w))

    # 10c. Row/col broadcast (same-size)
    if fixed_size is not None:
        h, w = fixed_size
        rb = detect_row_broadcast(train_pairs)
        if rb is not None:
            yield f"row_broadcast_{rb}", (
                lambda h=h, w=w, rb=rb: make_row_broadcast(h, w, rb)
            )
        cb = detect_col_broadcast(train_pairs)
        if cb is not None:
            yield f"col_broadcast_{cb}", (
                lambda h=h, w=w, cb=cb: make_col_broadcast(h, w, cb)
            )

    # 10d. Crop + color
    ccp = detect_crop_plus_color(train_pairs)
    if ccp is not None:
        r0, c0, oh_, ow_, cmap = ccp
        yield f"crop_color_{r0}_{c0}", (
            lambda r0=r0, c0=c0, oh_=oh_, ow_=ow_, cmap=cmap:
                make_crop_then_color(r0, c0, oh_, ow_, cmap)
        )

    # 10e. Border add
    if fixed_size is not None:
        ba = detect_border_add(train_pairs)
        if ba is not None:
            b, fill = ba
            h, w = fixed_size
            if h + 2*b <= H and w + 2*b <= W:
                def _build_border(h=h, w=w, b=b, fill=fill):
                    m = make_pad_with_border(h, w, b, fill)
                    if m is None:
                        raise ValueError("border add exceeds canvas")
                    return m
                yield f"border_add_{b}_{fill}", _build_border

    # 11. Constant output
    cst = detect_const(train_pairs)
    if cst is not None:
        yield "const", (lambda k=cst: make_const(k))


def solve_task(task_data: dict, task_name: str = "") -> Optional[Any]:
    train_pairs = task_data.get("train", []) or []
    test_pairs = task_data.get("test", []) or []
    arc_gen = task_data.get("arc-gen", []) or []
    arc_sample = arc_gen[:10] if arc_gen else []
    all_pairs = train_pairs + test_pairs + arc_sample
    if not all_pairs or not train_pairs:
        return None, None, None

    best_model = None
    best_cost = float("inf")
    best_label = None

    for label, builder in _candidates(train_pairs, all_pairs):
        try:
            model = builder()
        except Exception:
            continue
        if not check_fits(model) or not check_ops_allowed(model):
            continue
        if not check_model_correct(model, all_pairs):
            continue
        c = cost(model)
        if c < best_cost:
            best_cost = c
            best_model = model
            best_label = label

    return best_model, best_label, best_cost


# =============================================================================
# Driver: process all tasks, emit submission.zip
# =============================================================================

def load_task(path):
    with open(path) as f:
        return json.load(f)

def save_model(model, path):
    try:
        onnx.checker.check_model(model)
    except Exception:
        pass
    with open(path, "wb") as f:
        f.write(model.SerializeToString())


def find_task_files():
    candidates = [
        Path("/kaggle/input/competitions/neurogolf-2026"),
        Path("/kaggle/input/neurogolf-2026"),
        Path("/kaggle/input/arc-prize-2025/arc-agi-1-training"),
        Path("/kaggle/input"),
        Path("."),
    ]
    for d in candidates:
        if d.exists():
            files = sorted(d.glob("task*.json"))
            if files:
                return files
    task_files = []
    root_base = "/kaggle/input" if Path("/kaggle/input").exists() else "."
    for root, _, files in os.walk(root_base):
        for fname in files:
            if fname.startswith("task") and fname.endswith(".json"):
                task_files.append(Path(root) / fname)
    return sorted(task_files)


In [ ]:
import starter_solver as starter
import fast_solver as fast


def load_neurogolf_utils(task_dir: Path):
    utils_path = task_dir / 'neurogolf_utils' / 'neurogolf_utils.py'
    spec = importlib.util.spec_from_file_location('neurogolf_utils_local', utils_path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f'Unable to load helper module from {utils_path}')
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


UTILS = load_neurogolf_utils(TASK_DIR)


def full_public_check(namespace_module, model: onnx.ModelProto, task_data: dict) -> bool:
    all_pairs = task_data.get('train', []) + task_data.get('test', []) + task_data.get('arc-gen', [])
    try:
        session = ort.InferenceSession(model.SerializeToString(), providers=['CPUExecutionProvider'])
    except Exception:
        return False
    for pair in all_pairs:
        input_tensor = namespace_module.grid_to_tensor(pair['input'])
        output_grid = pair['output']
        output_h = len(output_grid)
        output_w = len(output_grid[0]) if output_grid else 0
        try:
            prediction = session.run(['output'], {'input': input_tensor})[0]
        except Exception:
            return False
        predicted_grid = namespace_module.tensor_to_grid(prediction, output_h, output_w)
        if predicted_grid != output_grid:
            return False
    return True


def validate_and_score_payload(filename: str, payload: bytes) -> dict:
    with tempfile.TemporaryDirectory() as tmpdir:
        path = Path(tmpdir) / filename
        path.write_bytes(payload)
        model = onnx.load(str(path))
        onnx.checker.check_model(model)
        session = ort.InferenceSession(str(path), providers=['CPUExecutionProvider'])
        sample = np.zeros((1, 10, 30, 30), dtype=np.float32)
        _ = session.run(['output'], {'input': sample})[0]
        macs, memory, params = UTILS.score_network(str(path))
        total_cost = int(macs + memory + params)
        return {
            'filesize_bytes': int(path.stat().st_size),
            'macs': int(macs),
            'memory': int(memory),
            'params': int(params),
            'total_cost': total_cost,
        }


def make_small_mask_gather_model(gather_indices: np.ndarray, mask_hw: np.ndarray, name: str) -> onnx.ModelProto:
    shape_chw = onh.from_array(np.array([1, 10, 900], dtype=np.int64), name='shape_chw')
    shape_out = onh.from_array(np.array([1, 10, 30, 30], dtype=np.int64), name='shape_out')
    gather_tensor = onh.from_array(gather_indices.astype(np.int64), name='gather_idx')
    mask_tensor = onh.from_array(mask_hw.astype(np.float32), name='mask')
    nodes = [
        oh.make_node('Reshape', ['input', 'shape_chw'], ['flat']),
        oh.make_node('Gather', ['flat', 'gather_idx'], ['flat_gathered'], axis=2),
        oh.make_node('Reshape', ['flat_gathered', 'shape_out'], ['gathered']),
        oh.make_node('Mul', ['gathered', 'mask'], ['output']),
    ]
    graph = oh.make_graph(
        nodes,
        name,
        [oh.make_tensor_value_info('input', onnx.TensorProto.FLOAT, [1, 10, 30, 30])],
        [oh.make_tensor_value_info('output', onnx.TensorProto.FLOAT, [1, 10, 30, 30])],
        [shape_chw, shape_out, gather_tensor, mask_tensor],
    )
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid('', 17)])
    model.ir_version = 8
    return model


def infer_zero_ambiguity_spatial(task_data: dict) -> tuple[onnx.ModelProto | None, int | None]:
    pairs = task_data.get('train', []) + task_data.get('test', []) + task_data.get('arc-gen', [])
    for pair in pairs:
        if (
            len(pair['input']) > 30
            or len(pair['input'][0]) > 30
            or len(pair['output']) > 30
            or len(pair['output'][0]) > 30
        ):
            return None, None

    inputs = []
    outputs = []
    for pair in pairs:
        inputs.append(starter.grid_to_tensor(pair['input'])[0].reshape(10, -1))
        outputs.append(starter.grid_to_tensor(pair['output'])[0].reshape(10, -1))
    inputs = np.stack(inputs, axis=0)
    outputs = np.stack(outputs, axis=0)

    input_lookup: dict[bytes, list[int]] = {}
    for source_index in range(900):
        signature = inputs[:, :, source_index].tobytes()
        input_lookup.setdefault(signature, []).append(source_index)

    gather_indices = np.zeros((900,), dtype=np.int64)
    mask_hw = np.zeros((1, 1, 30, 30), dtype=np.float32)
    ambiguous_positions = 0

    for output_index in range(900):
        signature_array = outputs[:, :, output_index]
        if np.all(signature_array == 0.0):
            continue
        candidates = input_lookup.get(signature_array.tobytes())
        if not candidates:
            return None, None
        if len(candidates) > 1:
            ambiguous_positions += 1
        source_index = candidates[0]
        gather_indices[output_index] = source_index
        row, col = divmod(output_index, 30)
        mask_hw[0, 0, row, col] = 1.0

    model = make_small_mask_gather_model(gather_indices, mask_hw, 'spatial_copy_zero_ambiguity')
    if not full_public_check(starter, model, task_data):
        return None, None
    return model, ambiguous_positions


def export_candidate(task_id: str, label: str, source: str, model: onnx.ModelProto, save_fn, extra: dict | None = None) -> dict:
    with tempfile.TemporaryDirectory() as tmpdir:
        path = Path(tmpdir) / f'{task_id}_{source}.onnx'
        save_fn(model, path)
        payload = path.read_bytes()
    details = validate_and_score_payload(f'{task_id}_{source}.onnx', payload)
    record = {
        'task_id': task_id,
        'label': label,
        'source': source,
        'payload': payload,
        **details,
    }
    if extra:
        record.update(extra)
    return record


def choose_best(candidates: list[dict]) -> dict | None:
    if not candidates:
        return None
    return min(candidates, key=lambda item: item['total_cost'])


task_paths = sorted(TASK_DIR.glob('task*.json'))
print(f'Found {len(task_paths)} tasks')

rows = []
solver_counts = Counter()
output_paths = []

for index, task_path in enumerate(task_paths, start=1):
    task_id = task_path.stem
    task_data = starter.load_task(task_path)
    candidates = []

    try:
        baseline_model = starter.solve_task(task_data, task_id)
        if baseline_model is not None and full_public_check(starter, baseline_model, task_data):
            candidates.append(export_candidate(task_id, 'starter', 'starter', baseline_model, starter.save_model))
    except Exception as exc:
        pass

    try:
        fast_output = fast.solve_task(task_data, task_id)
        if isinstance(fast_output, tuple):
            fast_model = fast_output[0]
            fast_label = str(fast_output[1]) if len(fast_output) > 1 else 'fast_public'
        else:
            fast_model = fast_output
            fast_label = 'fast_public'
        if fast_model is not None and full_public_check(fast, fast_model, task_data):
            candidates.append(export_candidate(task_id, fast_label, 'fast_public', fast_model, fast.save_model))
    except Exception:
        pass

    spatial_model, ambiguous_positions = infer_zero_ambiguity_spatial(task_data)
    if spatial_model is not None and ambiguous_positions == 0:
        candidates.append(
            export_candidate(
                task_id,
                'spatial_zero_ambiguity',
                'spatial_zero_ambiguity',
                spatial_model,
                starter.save_model,
                extra={'ambiguous_positions': ambiguous_positions},
            )
        )

    best = choose_best(candidates)
    if best is None:
        continue

    out_path = WORKDIR / f'{task_id}.onnx'
    out_path.write_bytes(best.pop('payload'))
    best['model_path'] = str(out_path)
    rows.append(best)
    output_paths.append(out_path)
    solver_counts[best['source']] += 1

    if index % 25 == 0 or index == len(task_paths):
        print(f'[{index:03d}/{len(task_paths):03d}] solved so far: {len(rows)}')

rows.sort(key=lambda row: row['task_id'])
submission_csv = WORKDIR / 'submission.csv'
fieldnames = []
for row in rows:
    for key in row.keys():
        if key not in fieldnames:
            fieldnames.append(key)
with submission_csv.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

submission_zip = WORKDIR / 'submission.zip'
with zipfile.ZipFile(submission_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for path in output_paths:
        zf.write(path, path.name)

proxy_score = sum(max(0.0, 25.0 - math.log(int(row['total_cost']))) for row in rows)
summary = {
    'task_count': len(rows),
    'proxy_score': proxy_score,
    'solver_counts': dict(solver_counts),
    'submission_csv': str(submission_csv),
    'submission_zip': str(submission_zip),
}
summary_path = WORKDIR / 'summary.json'
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))


In [ ]:
from IPython.display import FileLink, display
for rel in ['submission.zip', 'submission.csv', 'summary.json']:
    path = WORKDIR / rel
    if path.exists():
        display(FileLink(path))
